# LightCyto — final matched TinyViT ablation

**Four arms × three seeds = 12 student runs.** This notebook measures the separate contributions of your custom morphology-guided attention and SIPaKMeD knowledge distillation, using the exact split from `lightcyto_fresh_reports.zip`.

| Arm | Custom attention | SIPaKMeD KD | Training objective |
|---|---|---|---|
| A — TinyViT + shared head | No | No | CE |
| B — TinyViT + shared head + KD | No | Yes | CE + 0.25 KD |
| C — TinyViT + custom attention | Yes | No | CE |
| D — Full global model | Yes | Yes | CE + 0.25 KD |

**Fixed seeds:** 42, 123, 2026. **Student budget:** 45 epochs each. **Teacher:** one ConvNeXt-Tiny, trained for 12 epochs or restored from the exactly verified prior run. **Input:** 224 × 224.

All arms start from the same TinyViT ImageNet-pretrained backbone and paired common projection/classifier tensors within each seed. The pretrained checkpoint itself used ImageNet distillation; “no KD” above means **no additional SIPaKMeD teacher loss**, not absence of distillation in pretraining.

Outputs include train/validation histories, locked test predictions, mean ± seed SD, source-group bootstrap confidence intervals, paired factorial effects, confusion matrices, model weights, parameter counts, matrix MAC estimates and same-device latency.

**Scope:** this is a post-selection follow-up on a previously inspected dataset/test partition. It is not external validation or five-fold CV. There is no PPO, crop fusion, RKD, TTA or ensemble in this controlled global-model ablation.

## 1. Kaggle setup

1. Upload this `.ipynb` into Kaggle. Enable **GPU** and **Internet** for package/pretrained-weight downloads. The notebook uses **one GPU (`cuda:0`)** for every arm, even if two are available.
2. Attach the original SIPaKMeD image dataset containing all five `im_*` folders and their original `CROPPED` images/source fields.
3. Attach your **My_final_model** input containing `lightcyto_tinyvit_end_to_end/teacher/best.pt` and `experiment.json`. This version is configured to reuse that verified teacher. The reports ZIP alone has no weights.
4. Leave `dataset_root=''`; the teacher source is already set to `My_final_model/lightcyto_tinyvit_end_to_end`. **Run All**. The source-check cell resolves Kaggle's actual mount path, including normalized or nested paths. If multiple image datasets are found, set `dataset_root` to the folder directly containing the five class folders.

No separate split CSV upload is needed. The original source-group assignments are included below as readable CSV text, with the original full-manifest fingerprint. No encoded Python payloads or executable compressed data are used.

Do not change the training configuration after a run has begun. Changing paths/session time is allowed; changing scientific settings or package versions requires a separate output folder. A Kaggle **dataset mount failure before cell execution** must be resolved by reattaching the input; notebook code cannot repair an unmounted input.

In [1]:
import importlib.metadata, importlib.util, subprocess, sys
PINNED_TIMM = '1.0.22'
PINNED_FVCORE = '0.1.5.post20221221'
for package, version, no_deps in [('timm', PINNED_TIMM, True), ('fvcore', PINNED_FVCORE, False)]:
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        if package in sys.modules:
            raise RuntimeError('Restart the kernel before changing an imported package: '+package)
        command = [sys.executable, '-m', 'pip', 'install', '--quiet']
        if no_deps:
            command.append('--no-deps')
        subprocess.check_call(command+[package+'=='+version])
missing = [x for x in ('huggingface_hub','safetensors') if importlib.util.find_spec(x) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
# Keep Kaggle's installed CUDA-compatible torch/torchvision builds.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.6 MB/s eta 0:00:00


## Imports

In [2]:
from __future__ import annotations
import os, sys, re, io, json, math, time, copy, random, hashlib, shutil, zipfile, gc, inspect, platform, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import timm
import sklearn
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix
from fvcore.nn import FlopCountAnalysis

## Configuration

Defaults already run the complete study, including testing after validation lock.

- `dataset_root`: original image dataset, not the reports folder.
- `fresh_source`: set to your screenshot's **My_final_model/lightcyto_tinyvit_end_to_end** input/folder. The source-check cell resolves the actual path and verifies `teacher/best.pt` before training. Set `reuse_verified_fresh=False` only if you intentionally want a new teacher.
- `resume_source`: full output of **this ablation notebook**, for continuing an interrupted session.
- `output_root`: writable destination under `/kaggle/working/`.
- `run_final_test=True`: test all four predeclared arms after all 12 validation results are complete.
- `session_hours=10`: pause at an epoch/stage boundary and preserve progress. Actual wall time depends on the GPU; each epoch prints an ETA.

All 12 students are newly trained for the paired comparison. Historical student results are not mixed into the new three-seed mean. Completed jobs from this notebook are reused on resume.

In [3]:
@dataclass
class Config:
    dataset_root: str = ''  # Folder directly containing the five im_* class folders; blank auto-discovers.
    output_root: str = '/kaggle/working/lightcyto_final_ablation'
    resume_source: str = ''  # This ablation notebook's FULL saved output, for a new Kaggle session.
    fresh_source: str = 'My_final_model/lightcyto_tinyvit_end_to_end'  # Screenshot input/folder; resolved before Run.
    reuse_verified_fresh: bool = True
    seeds: tuple = (42, 123, 2026)
    teacher_seed: int = 7001
    image_size: int = 224
    feature_dim: int = 192
    attention_heads: int = 4
    batch_size: int = 16
    teacher_batch_size: int = 8
    num_workers: int = 0
    teacher_epochs: int = 12
    student_epochs: int = 45
    teacher_lr: float = 1e-4
    student_lr: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = .05
    kd_weight: float = .25
    kd_temperature: float = 2.
    pretrained: bool = True
    amp: bool = True
    allow_cpu: bool = False
    session_hours: float = 10.
    run_final_test: bool = True
    bootstrap_replicates: int = 2000
    bootstrap_seed: int = 91723
    benchmark_images: int = 64
    benchmark_repeats: int = 3
    benchmark_warmup: int = 20

CFG = Config()
CLASSES = ('Superficial-Intermediate','Parabasal','Metaplastic','Koilocytotic','Dyskeratotic')
STUDENT_NAME = 'tiny_vit_5m_224.dist_in22k_ft_in1k'
VERSION = 'lightcyto-tinyvit-attention-kd-factorial-v1'
ARMS = {
    'A_PLAIN': {'attention': False, 'kd': False, 'label': 'TinyViT + shared head'},
    'B_KD': {'attention': False, 'kd': True, 'label': 'TinyViT + shared head + KD'},
    'C_ATTENTION': {'attention': True, 'kd': False, 'label': 'TinyViT + custom attention'},
    'D_FULL': {'attention': True, 'kd': True, 'label': 'TinyViT + custom attention + KD'},
}
PRIMARY_ARM = 'D_FULL'  # Fixed before this ablation; never selected by test score.
MEAN = torch.tensor([.485,.456,.406]).view(1,3,1,1)
STD = torch.tensor([.229,.224,.225]).view(1,3,1,1)

## Original split and source identity

**Exact original partition:** 2,568 training images, 719 validation images and 762 test images, out of 4,049 unique cells. This code stores source-group assignments, not image pixels.

The manifest is reconstructed from the original files and verified against the original checksum. A changed dataset is rejected rather than silently receiving a new split. Grouping is by source image plus exact-duplicate unions; patient identifiers are unverified. The test partition was previously examined during this project, which must be disclosed in reporting.

In [4]:
ORIGINAL_EXPERIMENT = json.loads(r'''{
  "version": "tinyvit-kd-shared-crops-ppo-fresh-v1",
  "implementation_sha256": "fe3381c735731426da8b256415d90e3ec5962011c233350f308a5ce58f7cb8ee",
  "config": {
    "seeds": [
      42
    ],
    "split_seed": 42,
    "teacher_seed": 7001,
    "image_size": 224,
    "feature_dim": 192,
    "attention_heads": 4,
    "batch_size": 16,
    "teacher_batch_size": 8,
    "num_workers": 0,
    "teacher_epochs": 12,
    "student_epochs": 45,
    "fusion_epochs": 30,
    "ppo_rounds": 30,
    "teacher_lr": 0.0001,
    "student_lr": 0.0003,
    "fusion_lr": 0.0003,
    "ppo_lr": 0.0002,
    "weight_decay": 0.0001,
    "label_smoothing": 0.05,
    "kd_weight": 0.25,
    "kd_temperature": 2.0,
    "crop_fraction": 0.6,
    "overlap_penalty": 0.5,
    "crop_cost": 0.025,
    "max_crops": 3,
    "ppo_epochs": 4,
    "ppo_batch_size": 256,
    "ppo_minibatch": 128,
    "ppo_clip": 0.2,
    "gae_lambda": 0.95,
    "gamma": 1.0,
    "entropy_weight": 0.01,
    "value_weight": 0.5,
    "target_kl": 0.02,
    "confidence_threshold": 0.8,
    "grad_clip": 0.5,
    "pretrained": true,
    "amp": true
  },
  "dataset_sha256": "0211e7296e54bd429bcbc8eacc7e80f5c05affbfa3e4eda38ef37ec69fac1ddf",
  "runtime": {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "timm": "1.0.22",
    "numpy": "2.1.3",
    "pandas": "2.3.3"
  },
  "classes": [
    "Superficial-Intermediate",
    "Parabasal",
    "Metaplastic",
    "Koilocytotic",
    "Dyskeratotic"
  ],
  "teacher": "torchvision:convnext_tiny ImageNet1K_V1",
  "student": "tiny_vit_5m_224.dist_in22k_ft_in1k",
  "grouping": "source-image groups + exact duplicate union; patient IDs unverified",
  "evaluation_scope": "post-selection follow-up on a previously studied dataset; new source-group holdout",
  "contract_hash": "44a2ff126d481cd267bfd1b093b8670a8d35f60722c4a096877d98519f13e2c9"
}''')
ORIGINAL_TEACHER_SHA = 'ea668df5e4bc1036495a914f53ff75f966d68ae0adbba158ad3fd187689418e0'
ORIGINAL_STUDENT_SHA = 'af54686aa63f394a8da4007ee78ccc0f5a14cf042d1eac9f4c71a7e56a6daaca'
FROZEN_GROUP_SPLITS_CSV = r'''
group_id,split
Dyskeratotic:1,test
Dyskeratotic:10,test
Dyskeratotic:100,train
Dyskeratotic:101,train
Dyskeratotic:102,train
Dyskeratotic:103,train
Dyskeratotic:104,train
Dyskeratotic:105,train
Dyskeratotic:106,train
Dyskeratotic:107,train
Dyskeratotic:108,test
Dyskeratotic:109,train
Dyskeratotic:11,train
Dyskeratotic:110,val
Dyskeratotic:111,val
Dyskeratotic:112,train
Dyskeratotic:113,val
Dyskeratotic:114,test
Dyskeratotic:115,train
Dyskeratotic:116,train
Dyskeratotic:117,val
Dyskeratotic:118,train
Dyskeratotic:119,train
Dyskeratotic:12,val
Dyskeratotic:120,test
Dyskeratotic:121,train
Dyskeratotic:122,test
Dyskeratotic:123,train
Dyskeratotic:124,train
Dyskeratotic:125,train
Dyskeratotic:126,train
Dyskeratotic:127,train
Dyskeratotic:128,train
Dyskeratotic:129,train
Dyskeratotic:13,test
Dyskeratotic:130,train
Dyskeratotic:131,train
Dyskeratotic:132,train
Dyskeratotic:133,train
Dyskeratotic:134,train
Dyskeratotic:135,val
Dyskeratotic:136,val
Dyskeratotic:137,train
Dyskeratotic:138,val
Dyskeratotic:139,train
Dyskeratotic:14,test
Dyskeratotic:140,train
Dyskeratotic:141,train
Dyskeratotic:142,val
Dyskeratotic:143,train
Dyskeratotic:144,test
Dyskeratotic:145,train
Dyskeratotic:146,train
Dyskeratotic:147,test
Dyskeratotic:148,test
Dyskeratotic:149,test
Dyskeratotic:15,train
Dyskeratotic:150,train
Dyskeratotic:151,val
Dyskeratotic:152,test
Dyskeratotic:153,train
Dyskeratotic:154,train
Dyskeratotic:155,train
Dyskeratotic:156,train
Dyskeratotic:157,train
Dyskeratotic:158,test
Dyskeratotic:159,train
Dyskeratotic:16,train
Dyskeratotic:160,train
Dyskeratotic:161,train
Dyskeratotic:162,val
Dyskeratotic:163,test
Dyskeratotic:164,val
Dyskeratotic:165,test
Dyskeratotic:166,test
Dyskeratotic:167,train
Dyskeratotic:168,train
Dyskeratotic:169,train
Dyskeratotic:17,train
Dyskeratotic:170,train
Dyskeratotic:171,train
Dyskeratotic:172,val
Dyskeratotic:173,train
Dyskeratotic:174,val
Dyskeratotic:175,train
Dyskeratotic:176,train
Dyskeratotic:177,test
Dyskeratotic:178,test
Dyskeratotic:179,train
Dyskeratotic:18,train
Dyskeratotic:180,test
Dyskeratotic:181,train
Dyskeratotic:182,val
Dyskeratotic:183,val
Dyskeratotic:184,train
Dyskeratotic:185,train
Dyskeratotic:186,val
Dyskeratotic:187,train
Dyskeratotic:188,train
Dyskeratotic:189,train
Dyskeratotic:19,train
Dyskeratotic:190,val
Dyskeratotic:191,train
Dyskeratotic:192,train
Dyskeratotic:193,train
Dyskeratotic:194,test
Dyskeratotic:195,train
Dyskeratotic:196,train
Dyskeratotic:197,train
Dyskeratotic:198,train
Dyskeratotic:199,train
Dyskeratotic:2,train
Dyskeratotic:20,train
Dyskeratotic:200,train
Dyskeratotic:201,train
Dyskeratotic:202,train
Dyskeratotic:203,train
Dyskeratotic:204,train
Dyskeratotic:205,test
Dyskeratotic:206,test
Dyskeratotic:207,train
Dyskeratotic:208,test
Dyskeratotic:209,val
Dyskeratotic:21,train
Dyskeratotic:210,val
Dyskeratotic:211,test
Dyskeratotic:212,val
Dyskeratotic:213,train
Dyskeratotic:214,val
Dyskeratotic:215,train
Dyskeratotic:216,train
Dyskeratotic:217,train
Dyskeratotic:218,val
Dyskeratotic:219,test
Dyskeratotic:22,train
Dyskeratotic:220,test
Dyskeratotic:221,train
Dyskeratotic:222,test
Dyskeratotic:223,train
Dyskeratotic:23,train
Dyskeratotic:24,test
Dyskeratotic:25,train
Dyskeratotic:26,train
Dyskeratotic:27,test
Dyskeratotic:28,train
Dyskeratotic:29,train
Dyskeratotic:3,train
Dyskeratotic:30,val
Dyskeratotic:31,train
Dyskeratotic:32,test
Dyskeratotic:33,train
Dyskeratotic:34,test
Dyskeratotic:35,test
Dyskeratotic:36,train
Dyskeratotic:37,train
Dyskeratotic:38,train
Dyskeratotic:39,test
Dyskeratotic:4,train
Dyskeratotic:40,train
Dyskeratotic:41,test
Dyskeratotic:42,val
Dyskeratotic:43,train
Dyskeratotic:44,train
Dyskeratotic:45,train
Dyskeratotic:46,train
Dyskeratotic:47,train
Dyskeratotic:48,train
Dyskeratotic:49,test
Dyskeratotic:5,val
Dyskeratotic:50,train
Dyskeratotic:51,train
Dyskeratotic:52,train
Dyskeratotic:53,train
Dyskeratotic:54,train
Dyskeratotic:55,val
Dyskeratotic:56,train
Dyskeratotic:57,train
Dyskeratotic:58,val
Dyskeratotic:59,train
Dyskeratotic:6,train
Dyskeratotic:60,train
Dyskeratotic:61,test
Dyskeratotic:62,val
Dyskeratotic:63,train
Dyskeratotic:64,train
Dyskeratotic:65,train
Dyskeratotic:66,train
Dyskeratotic:67,train
Dyskeratotic:68,train
Dyskeratotic:69,val
Dyskeratotic:7,val
Dyskeratotic:70,train
Dyskeratotic:71,train
Dyskeratotic:72,test
Dyskeratotic:73,train
Dyskeratotic:74,test
Dyskeratotic:75,train
Dyskeratotic:76,train
Dyskeratotic:77,val
Dyskeratotic:78,test
Dyskeratotic:79,val
Dyskeratotic:8,val
Dyskeratotic:80,train
Dyskeratotic:81,train
Dyskeratotic:82,train
Dyskeratotic:83,test
Dyskeratotic:84,val
Dyskeratotic:85,train
Dyskeratotic:86,test
Dyskeratotic:87,train
Dyskeratotic:88,train
Dyskeratotic:89,test
Dyskeratotic:9,train
Dyskeratotic:90,train
Dyskeratotic:91,train
Dyskeratotic:92,train
Dyskeratotic:93,train
Dyskeratotic:94,val
Dyskeratotic:95,train
Dyskeratotic:96,val
Dyskeratotic:97,train
Dyskeratotic:98,train
Dyskeratotic:99,train
Koilocytotic:1,test
Koilocytotic:10,test
Koilocytotic:100,test
Koilocytotic:101,train
Koilocytotic:102,train
Koilocytotic:103,train
Koilocytotic:104,test
Koilocytotic:105,train
Koilocytotic:106,train
Koilocytotic:107,train
Koilocytotic:108,val
Koilocytotic:109,train
Koilocytotic:11,val
Koilocytotic:110,train
Koilocytotic:111,train
Koilocytotic:112,train
Koilocytotic:113,train
Koilocytotic:114,val
Koilocytotic:115,val
Koilocytotic:116,train
Koilocytotic:117,train
Koilocytotic:118,train
Koilocytotic:119,test
Koilocytotic:12,train
Koilocytotic:120,train
Koilocytotic:121,train
Koilocytotic:122,val
Koilocytotic:123,train
Koilocytotic:124,test
Koilocytotic:125,train
Koilocytotic:126,val
Koilocytotic:127,test
Koilocytotic:128,train
Koilocytotic:129,test
Koilocytotic:13,train
Koilocytotic:130,train
Koilocytotic:131,train
Koilocytotic:132,test
Koilocytotic:133,train
Koilocytotic:134,train
Koilocytotic:135,train
Koilocytotic:136,test
Koilocytotic:137,test
Koilocytotic:138,train
Koilocytotic:139,test
Koilocytotic:14,train
Koilocytotic:140,test
Koilocytotic:141,val
Koilocytotic:142,train
Koilocytotic:143,train
Koilocytotic:144,train
Koilocytotic:145,val
Koilocytotic:146,test
Koilocytotic:147,val
Koilocytotic:148,train
Koilocytotic:149,train
Koilocytotic:15,train
Koilocytotic:150,test
Koilocytotic:151,train
Koilocytotic:152,train
Koilocytotic:153,train
Koilocytotic:154,train
Koilocytotic:155,train
Koilocytotic:156,train
Koilocytotic:157,train
Koilocytotic:158,train
Koilocytotic:159,test
Koilocytotic:16,train
Koilocytotic:160,val
Koilocytotic:161,test
Koilocytotic:162,val
Koilocytotic:163,train
Koilocytotic:164,test
Koilocytotic:165,train
Koilocytotic:166,train
Koilocytotic:167,train
Koilocytotic:168,train
Koilocytotic:169,train
Koilocytotic:17,val
Koilocytotic:170,train
Koilocytotic:171,train
Koilocytotic:172,train
Koilocytotic:173,val
Koilocytotic:174,test
Koilocytotic:175,train
Koilocytotic:176,train
Koilocytotic:177,val
Koilocytotic:178,train
Koilocytotic:179,train
Koilocytotic:18,test
Koilocytotic:180,train
Koilocytotic:181,test
Koilocytotic:182,train
Koilocytotic:183,train
Koilocytotic:184,test
Koilocytotic:185,train
Koilocytotic:186,train
Koilocytotic:187,train
Koilocytotic:188,train
Koilocytotic:189,train
Koilocytotic:19,test
Koilocytotic:190,train
Koilocytotic:191,train
Koilocytotic:192,train
Koilocytotic:193,train
Koilocytotic:194,train
Koilocytotic:195,train
Koilocytotic:196,val
Koilocytotic:197,train
Koilocytotic:198,train
Koilocytotic:199,train
Koilocytotic:2,val
Koilocytotic:20,test
Koilocytotic:200,train
Koilocytotic:201,test
Koilocytotic:202,train
Koilocytotic:203,test
Koilocytotic:204,test
Koilocytotic:205,test
Koilocytotic:206,val
Koilocytotic:207,train
Koilocytotic:208,train
Koilocytotic:209,train
Koilocytotic:21,val
Koilocytotic:210,train
Koilocytotic:211,train
Koilocytotic:212,test
Koilocytotic:213,train
Koilocytotic:214,train
Koilocytotic:215,test
Koilocytotic:216,train
Koilocytotic:217,test
Koilocytotic:218,train
Koilocytotic:219,test
Koilocytotic:22,val
Koilocytotic:220,val
Koilocytotic:221,train
Koilocytotic:222,train
Koilocytotic:223,train
Koilocytotic:224,train
Koilocytotic:225,val
Koilocytotic:226,train
Koilocytotic:227,train
Koilocytotic:228,test
Koilocytotic:229,train
Koilocytotic:23,train
Koilocytotic:230,test
Koilocytotic:231,train
Koilocytotic:232,train
Koilocytotic:233,train
Koilocytotic:234,train
Koilocytotic:235,train
Koilocytotic:236,val
Koilocytotic:237,train
Koilocytotic:238,train
Koilocytotic:24,train
Koilocytotic:25,test
Koilocytotic:26,train
Koilocytotic:27,train
Koilocytotic:28,test
Koilocytotic:29,train
Koilocytotic:3,train
Koilocytotic:30,val
Koilocytotic:31,train
Koilocytotic:32,train
Koilocytotic:33,test
Koilocytotic:34,train
Koilocytotic:35,val
Koilocytotic:36,train
Koilocytotic:37,train
Koilocytotic:38,train
Koilocytotic:39,train
Koilocytotic:4,train
Koilocytotic:40,train
Koilocytotic:41,test
Koilocytotic:42,val
Koilocytotic:43,train
Koilocytotic:44,test
Koilocytotic:45,test
Koilocytotic:46,train
Koilocytotic:47,train
Koilocytotic:48,train
Koilocytotic:49,val
Koilocytotic:5,test
Koilocytotic:50,test
Koilocytotic:51,val
Koilocytotic:52,val
Koilocytotic:53,train
Koilocytotic:54,train
Koilocytotic:55,train
Koilocytotic:56,train
Koilocytotic:57,test
Koilocytotic:58,train
Koilocytotic:59,train
Koilocytotic:6,train
Koilocytotic:60,test
Koilocytotic:61,train
Koilocytotic:62,test
Koilocytotic:63,train
Koilocytotic:64,train
Koilocytotic:65,test
Koilocytotic:66,train
Koilocytotic:67,test
Koilocytotic:68,train
Koilocytotic:69,test
Koilocytotic:7,train
Koilocytotic:70,val
Koilocytotic:71,val
Koilocytotic:72,train
Koilocytotic:73,train
Koilocytotic:74,train
Koilocytotic:75,train
Koilocytotic:76,train
Koilocytotic:77,val
Koilocytotic:78,train
Koilocytotic:79,train
Koilocytotic:8,train
Koilocytotic:80,train
Koilocytotic:81,train
Koilocytotic:82,train
Koilocytotic:83,val
Koilocytotic:84,train
Koilocytotic:85,val
Koilocytotic:86,train
Koilocytotic:87,train
Koilocytotic:88,train
Koilocytotic:89,val
Koilocytotic:9,train
Koilocytotic:90,test
Koilocytotic:91,val
Koilocytotic:92,val
Koilocytotic:93,train
Koilocytotic:94,train
Koilocytotic:95,test
Koilocytotic:96,test
Koilocytotic:97,val
Koilocytotic:98,train
Koilocytotic:99,train
Metaplastic:1,train
Metaplastic:10,train
Metaplastic:100,val
Metaplastic:101,train
Metaplastic:102,train
Metaplastic:103,train
Metaplastic:104,train
Metaplastic:105,train
Metaplastic:106,test
Metaplastic:107,train
Metaplastic:108,test
Metaplastic:109,train
Metaplastic:11,train
Metaplastic:110,train
Metaplastic:111,train
Metaplastic:112,train
Metaplastic:113,train
Metaplastic:114,test
Metaplastic:115,train
Metaplastic:116,val
Metaplastic:117,train
Metaplastic:118,train
Metaplastic:119,test
Metaplastic:12,train
Metaplastic:120,train
Metaplastic:121,train
Metaplastic:122,train
Metaplastic:123,test
Metaplastic:124,train
Metaplastic:125,val
Metaplastic:126,train
Metaplastic:127,val
Metaplastic:128,train
Metaplastic:129,train
Metaplastic:13,train
Metaplastic:130,test
Metaplastic:131,train
Metaplastic:132,train
Metaplastic:133,train
Metaplastic:134,train
Metaplastic:135,val
Metaplastic:136,test
Metaplastic:137,train
Metaplastic:138,val
Metaplastic:139,train
Metaplastic:14,train
Metaplastic:140,train
Metaplastic:141,test
Metaplastic:142,val
Metaplastic:143,test
Metaplastic:144,val
Metaplastic:145,train
Metaplastic:146,test
Metaplastic:147,test
Metaplastic:148,train
Metaplastic:149,test
Metaplastic:15,train
Metaplastic:150,train
Metaplastic:151,train
Metaplastic:152,train
Metaplastic:153,train
Metaplastic:154,train
Metaplastic:155,val
Metaplastic:156,train
Metaplastic:157,test
Metaplastic:158,val
Metaplastic:159,train
Metaplastic:16,val
Metaplastic:160,test
Metaplastic:161,train
Metaplastic:162,train
Metaplastic:163,test
Metaplastic:164,train
Metaplastic:165,train
Metaplastic:166,train
Metaplastic:167,test
Metaplastic:168,train
Metaplastic:169,train
Metaplastic:17,val
Metaplastic:170,test
Metaplastic:171,test
Metaplastic:172,train
Metaplastic:173,train
Metaplastic:174,train
Metaplastic:175,train
Metaplastic:176,train
Metaplastic:177,test
Metaplastic:178,train
Metaplastic:179,test
Metaplastic:18,val
Metaplastic:180,train
Metaplastic:181,train
Metaplastic:182,train
Metaplastic:183,test
Metaplastic:184,train
Metaplastic:185,test
Metaplastic:186,train
Metaplastic:187,test
Metaplastic:188,train
Metaplastic:189,train
Metaplastic:19,train
Metaplastic:190,val
Metaplastic:191,train
Metaplastic:192,val
Metaplastic:193,train
Metaplastic:194,train
Metaplastic:195,test
Metaplastic:196,train
Metaplastic:197,train
Metaplastic:198,train
Metaplastic:199,val
Metaplastic:2,train
Metaplastic:20,val
Metaplastic:200,test
Metaplastic:201,val
Metaplastic:202,test
Metaplastic:203,train
Metaplastic:204,train
Metaplastic:205,train
Metaplastic:206,train
Metaplastic:207,train
Metaplastic:208,train
Metaplastic:209,val
Metaplastic:21,val
Metaplastic:210,train
Metaplastic:211,val
Metaplastic:212,train
Metaplastic:213,train
Metaplastic:214,train
Metaplastic:215,train
Metaplastic:216,train
Metaplastic:217,train
Metaplastic:218,val
Metaplastic:219,test
Metaplastic:22,train
Metaplastic:220,train
Metaplastic:221,train
Metaplastic:222,train
Metaplastic:223,val
Metaplastic:224,train
Metaplastic:225,test
Metaplastic:226,test
Metaplastic:227,train
Metaplastic:228,test
Metaplastic:229,train
Metaplastic:23,train
Metaplastic:230,test
Metaplastic:231,train
Metaplastic:232,train
Metaplastic:233,train
Metaplastic:234,val
Metaplastic:235,train
Metaplastic:236,train
Metaplastic:237,train
Metaplastic:238,val
Metaplastic:239,train
Metaplastic:24,train
Metaplastic:240,test
Metaplastic:241,train
Metaplastic:242,train
Metaplastic:243,train
Metaplastic:244,val
Metaplastic:245,train
Metaplastic:246,test
Metaplastic:247,train
Metaplastic:248,train
Metaplastic:249,train
Metaplastic:25,val
Metaplastic:250,train
Metaplastic:251,test
Metaplastic:252,train
Metaplastic:253,train
Metaplastic:254,test
Metaplastic:255,train
Metaplastic:256,train
Metaplastic:257,train
Metaplastic:258,train
Metaplastic:259,train
Metaplastic:26,train
Metaplastic:260,train
Metaplastic:261,train
Metaplastic:262,test
Metaplastic:263,test
Metaplastic:264,val
Metaplastic:265,test
Metaplastic:266,test
Metaplastic:267,train
Metaplastic:268,test
Metaplastic:269,train
Metaplastic:27,train
Metaplastic:270,train
Metaplastic:271,train
Metaplastic:28,val
Metaplastic:29,train
Metaplastic:3,train
Metaplastic:30,test
Metaplastic:31,train
Metaplastic:32,train
Metaplastic:33,train
Metaplastic:34,train
Metaplastic:35,train
Metaplastic:36,train
Metaplastic:37,val
Metaplastic:38,train
Metaplastic:39,test
Metaplastic:4,test
Metaplastic:40,train
Metaplastic:41,train
Metaplastic:42,train
Metaplastic:43,train
Metaplastic:44,train
Metaplastic:45,test
Metaplastic:46,val
Metaplastic:47,train
Metaplastic:48,train
Metaplastic:49,val
Metaplastic:5,test
Metaplastic:50,test
Metaplastic:51,train
Metaplastic:52,train
Metaplastic:53,train
Metaplastic:54,test
Metaplastic:55,train
Metaplastic:56,val
Metaplastic:57,train
Metaplastic:58,train
Metaplastic:59,train
Metaplastic:6,train
Metaplastic:60,train
Metaplastic:61,val
Metaplastic:62,test
Metaplastic:63,val
Metaplastic:64,train
Metaplastic:65,val
Metaplastic:66,val
Metaplastic:67,val
Metaplastic:68,test
Metaplastic:69,train
Metaplastic:7,val
Metaplastic:70,train
Metaplastic:71,train
Metaplastic:72,train
Metaplastic:73,test
Metaplastic:74,val
Metaplastic:75,train
Metaplastic:76,train
Metaplastic:77,test
Metaplastic:78,train
Metaplastic:79,test
Metaplastic:8,train
Metaplastic:80,val
Metaplastic:81,train
Metaplastic:82,train
Metaplastic:83,train
Metaplastic:84,train
Metaplastic:85,test
Metaplastic:86,train
Metaplastic:87,train
Metaplastic:88,train
Metaplastic:89,train
Metaplastic:9,train
Metaplastic:90,train
Metaplastic:91,test
Metaplastic:92,train
Metaplastic:93,train
Metaplastic:94,test
Metaplastic:95,train
Metaplastic:96,train
Metaplastic:97,val
Metaplastic:98,train
Metaplastic:99,train
Parabasal:1,train
Parabasal:10,train
Parabasal:100,train
Parabasal:101,train
Parabasal:102,val
Parabasal:103,train
Parabasal:104,test
Parabasal:105,train
Parabasal:106,train
Parabasal:107,train
Parabasal:108,test
Parabasal:11,train
Parabasal:12,val
Parabasal:13,train
Parabasal:14,test
Parabasal:15,train
Parabasal:16,train
Parabasal:17,train
Parabasal:18,train
Parabasal:19,train
Parabasal:2,train
Parabasal:20,train
Parabasal:21,train
Parabasal:22,train
Parabasal:23,train
Parabasal:24,val
Parabasal:25,train
Parabasal:26,train
Parabasal:27,train
Parabasal:28,train
Parabasal:29,test
Parabasal:3,train
Parabasal:30,train
Parabasal:31,val
Parabasal:32,val
Parabasal:33,train
Parabasal:34,train
Parabasal:35,val
Parabasal:36,train
Parabasal:37,train
Parabasal:38,train
Parabasal:39,train
Parabasal:4,train
Parabasal:40,train
Parabasal:41,train
Parabasal:42,train
Parabasal:43,train
Parabasal:44,test
Parabasal:45,train
Parabasal:46,train
Parabasal:47,test
Parabasal:48,test
Parabasal:49,train
Parabasal:5,train
Parabasal:50,test
Parabasal:51,train
Parabasal:52,train
Parabasal:53,train
Parabasal:54,train
Parabasal:55,train
Parabasal:56,train
Parabasal:57,val
Parabasal:58,train
Parabasal:59,train
Parabasal:6,train
Parabasal:60,val
Parabasal:61,val
Parabasal:62,test
Parabasal:63,test
Parabasal:64,train
Parabasal:65,train
Parabasal:66,train
Parabasal:67,val
Parabasal:68,train
Parabasal:69,val
Parabasal:7,train
Parabasal:70,train
Parabasal:71,train
Parabasal:72,val
Parabasal:73,train
Parabasal:74,train
Parabasal:75,train
Parabasal:76,train
Parabasal:77,train
Parabasal:78,train
Parabasal:79,val
Parabasal:8,train
Parabasal:80,test
Parabasal:81,train
Parabasal:82,train
Parabasal:83,train
Parabasal:84,val
Parabasal:85,train
Parabasal:86,test
Parabasal:87,test
Parabasal:88,train
Parabasal:89,train
Parabasal:9,test
Parabasal:90,train
Parabasal:91,test
Parabasal:92,train
Parabasal:93,test
Parabasal:94,train
Parabasal:95,train
Parabasal:96,train
Parabasal:97,val
Parabasal:98,train
Parabasal:99,train
Superficial-Intermediate:1,train
Superficial-Intermediate:10,val
Superficial-Intermediate:100,val
Superficial-Intermediate:101,train
Superficial-Intermediate:102,test
Superficial-Intermediate:103,test
Superficial-Intermediate:104,train
Superficial-Intermediate:105,train
Superficial-Intermediate:106,train
Superficial-Intermediate:107,test
Superficial-Intermediate:108,train
Superficial-Intermediate:109,train
Superficial-Intermediate:11,train
Superficial-Intermediate:110,test
Superficial-Intermediate:111,val
Superficial-Intermediate:112,test
Superficial-Intermediate:113,train
Superficial-Intermediate:114,train
Superficial-Intermediate:115,train
Superficial-Intermediate:116,test
Superficial-Intermediate:117,train
Superficial-Intermediate:118,val
Superficial-Intermediate:119,test
Superficial-Intermediate:12,train
Superficial-Intermediate:120,test
Superficial-Intermediate:121,train
Superficial-Intermediate:122,train
Superficial-Intermediate:123,train
Superficial-Intermediate:124,train
Superficial-Intermediate:125,test
Superficial-Intermediate:126,train
Superficial-Intermediate:13,train
Superficial-Intermediate:14,val
Superficial-Intermediate:15,train
Superficial-Intermediate:16,train
Superficial-Intermediate:17,train
Superficial-Intermediate:18,train
Superficial-Intermediate:19,train
Superficial-Intermediate:2,test
Superficial-Intermediate:20,train
Superficial-Intermediate:21,train
Superficial-Intermediate:22,test
Superficial-Intermediate:23,train
Superficial-Intermediate:24,train
Superficial-Intermediate:25,train
Superficial-Intermediate:26,test
Superficial-Intermediate:27,val
Superficial-Intermediate:28,train
Superficial-Intermediate:29,train
Superficial-Intermediate:3,val
Superficial-Intermediate:30,val
Superficial-Intermediate:31,train
Superficial-Intermediate:32,train
Superficial-Intermediate:33,train
Superficial-Intermediate:34,test
Superficial-Intermediate:35,train
Superficial-Intermediate:36,train
Superficial-Intermediate:37,train
Superficial-Intermediate:38,train
Superficial-Intermediate:39,val
Superficial-Intermediate:4,val
Superficial-Intermediate:40,val
Superficial-Intermediate:41,train
Superficial-Intermediate:42,train
Superficial-Intermediate:43,test
Superficial-Intermediate:44,test
Superficial-Intermediate:45,train
Superficial-Intermediate:46,train
Superficial-Intermediate:47,train
Superficial-Intermediate:48,test
Superficial-Intermediate:49,train
Superficial-Intermediate:5,train
Superficial-Intermediate:50,test
Superficial-Intermediate:51,test
Superficial-Intermediate:52,val
Superficial-Intermediate:53,train
Superficial-Intermediate:54,train
Superficial-Intermediate:55,val
Superficial-Intermediate:56,train
Superficial-Intermediate:57,train
Superficial-Intermediate:58,test
Superficial-Intermediate:59,train
Superficial-Intermediate:6,val
Superficial-Intermediate:60,train
Superficial-Intermediate:61,train
Superficial-Intermediate:62,train
Superficial-Intermediate:63,train
Superficial-Intermediate:64,test
Superficial-Intermediate:65,test
Superficial-Intermediate:66,train
Superficial-Intermediate:67,val
Superficial-Intermediate:68,val
Superficial-Intermediate:69,train
Superficial-Intermediate:7,train
Superficial-Intermediate:70,train
Superficial-Intermediate:71,train
Superficial-Intermediate:72,test
Superficial-Intermediate:73,val
Superficial-Intermediate:74,val
Superficial-Intermediate:75,train
Superficial-Intermediate:76,test
Superficial-Intermediate:77,val
Superficial-Intermediate:78,train
Superficial-Intermediate:79,train
Superficial-Intermediate:8,train
Superficial-Intermediate:80,train
Superficial-Intermediate:81,test
Superficial-Intermediate:82,val
Superficial-Intermediate:83,train
Superficial-Intermediate:84,train
Superficial-Intermediate:85,train
Superficial-Intermediate:86,test
Superficial-Intermediate:87,train
Superficial-Intermediate:88,test
Superficial-Intermediate:89,train
Superficial-Intermediate:9,train
Superficial-Intermediate:90,test
Superficial-Intermediate:91,val
Superficial-Intermediate:92,train
Superficial-Intermediate:93,train
Superficial-Intermediate:94,val
Superficial-Intermediate:95,train
Superficial-Intermediate:96,train
Superficial-Intermediate:97,train
Superficial-Intermediate:98,train
Superficial-Intermediate:99,val
'''

## Utilities

Atomic checkpoints, safe tensor loading, finite-value metric checks, deterministic epoch seeding and immutable experiment identity. Resuming an interrupted partial epoch replays that epoch from the last completed checkpoint.

In [5]:
class SessionPause(RuntimeError): pass

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def json_value(x):
    if isinstance(x, dict): return {str(k):json_value(v) for k,v in x.items()}
    if isinstance(x, (tuple,list)): return [json_value(v) for v in x]
    if isinstance(x, (np.integer,np.floating)): return x.item()
    if isinstance(x, Path): return str(x)
    if isinstance(x, torch.Tensor): return x.detach().cpu().tolist()
    return x

def save_json(x, path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(json_value(x),indent=2,allow_nan=False)); os.replace(tmp,path)

def read_json(path): return json.loads(Path(path).read_text())

def object_hash(x): return hashlib.sha256(json.dumps(json_value(x),sort_keys=True).encode()).hexdigest()

def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def save_pt(x,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');torch.save(x,tmp);os.replace(tmp,path)

def load_pt(path): return torch.load(path,map_location='cpu',weights_only=True)

def cpu_state(model): return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def clear_bias_cache(model):
    for m in model.modules():
        if hasattr(m,'attention_bias_cache'): m.attention_bias_cache.clear()

def install_state(model,state):
    model.load_state_dict(state,strict=True);clear_bias_cache(model)

def budget(ctx):
    if time.monotonic() >= ctx['deadline']: raise SessionPause('Session budget reached; resume this full output.')

def metric_values(y,logits,crops=None):
    y=np.asarray(y,dtype=int);logits=np.asarray(logits,dtype=np.float64)
    assert len(y)>0 and logits.shape==(len(y),5) and np.isfinite(logits).all()
    p=np.exp(logits-logits.max(1,keepdims=True));p/=p.sum(1,keepdims=True);pred=p.argmax(1)
    result={'n':len(y),'correct':int((y==pred).sum()),'accuracy':float(accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=list(range(5)),average='macro',zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'ce':float(-np.log(p[np.arange(len(y)),y].clip(1e-12)).mean())}
    if crops is not None: result['mean_crops']=float(np.mean(crops))
    return result

def quality(m): return (m['macro_f1'],m['accuracy'],-m.get('mean_crops',0.),-m['ce'])

def freeze(model):
    model.eval()
    for p in model.parameters():p.requires_grad_(False)
    return model

def implementation_hash():
    # Functions/methods are source-resolvable in IPython; whole interactive classes may not be.
    definitions={}
    for name,obj in globals().copy().items():
        if name=='Config':continue  # Scientific config values are hashed separately.
        if getattr(obj,'__module__',None)!=__name__:continue
        try:
            if inspect.isfunction(obj):definitions[name]=inspect.getsource(obj).strip()
            elif inspect.isclass(obj):
                methods={}
                for key,value in vars(obj).items():
                    if isinstance(value,(staticmethod,classmethod)):value=value.__func__
                    if inspect.isfunction(value):methods[key]=inspect.getsource(value).strip()
                definitions[name]={'bases':[base.__name__ for base in obj.__bases__],'methods':methods}
        except (OSError,TypeError) as e:
            raise RuntimeError('Run the visible notebook definition cells normally before prepare().') from e
    return object_hash(definitions)

## Original data audit and preprocessing

Uses the original dataset audit and preprocessing code: original `CROPPED` files only, image/source-field hashes, source grouping, aspect-preserving white letterboxing, flips/90° rotations and the same mild colour/brightness augmentation.

The train partition alone supplies optimization batches. Validation chooses checkpoints. Test pixel hashing is used only to check integrity before final evaluation; no test labels are used for fitting or checkpoint selection.

In [6]:
def resolve_dataset(hint):
    if hint:
        p=Path(hint).expanduser()
        if not p.is_dir():raise FileNotFoundError(f'DATASET_ROOT does not exist: {p}')
        required={'im_'+c for c in CLASSES}
        if not required <= {x.name for x in p.iterdir() if x.is_dir()}:
            raise ValueError('dataset_root must directly contain all five im_* class folders.')
        return p
    root=Path('/kaggle/input');required={'im_'+c for c in CLASSES}
    matches=[]
    for d in root.rglob('im_Dyskeratotic'):
        if d.is_dir() and required <= {p.name for p in d.parent.iterdir() if p.is_dir()}:
            matches.append(d.parent)
    matches=sorted(set(matches))
    if len(matches)!=1:
        raise FileNotFoundError('Set CFG.dataset_root to the folder containing the five im_* class folders. '
                                f'Found {len(matches)} candidates: {matches}')
    return matches[0]

def scan_cells(root):
    rows=[]
    for p in sorted(root.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in {'.bmp','.png','.jpg','.jpeg','.tif','.tiff'}:continue
        rel=p.relative_to(root)
        if not any(part.upper()=='CROPPED' for part in rel.parts[:-1]):continue
        names={part[3:] for part in rel.parts[:-1] if part.startswith('im_') and part[3:] in CLASSES}
        if len(names)!=1:raise ValueError(f'Cannot identify one class for {rel}')
        cls=next(iter(names));m=re.fullmatch(r'(\d+)[_-](\d+)',p.stem)
        if not m:raise ValueError(f'Unknown source-image filename: {rel}. Supply original SIPaKMeD CROPPED folders; do not randomly split ambiguous crops.')
        with Image.open(p) as im:
            im=im.convert('RGB');arr=np.asarray(im)
            ph=hashlib.sha256(str(im.size).encode()+arr.tobytes()).hexdigest()
        rows.append({'relative_path':rel.as_posix(),'class_name':cls,'label':CLASSES.index(cls),
                     'source_group':f'{cls}:{int(m.group(1))}', 'pixel_sha256':ph,'file_sha256':file_hash(p),
                     'sample_id':hashlib.sha256(rel.as_posix().encode()).hexdigest()[:20]})
    if not rows:raise ValueError('No cell images found under CROPPED directories.')
    frame=pd.DataFrame(rows)
    if set(frame.class_name)!=set(CLASSES):raise ValueError('All five SIPaKMeD classes are required.')
    # Verify available parent-field images as well as cell crops. Repeated source fields
    # under different IDs must stay in the same partition even if their crops differ.
    source_hashes={}
    for group,sub in frame.groupby('source_group',sort=False):
        cell=root/sub.iloc[0].relative_path
        stem=re.fullmatch(r'(\d+)[_-](\d+)',cell.stem).group(1)
        candidates=[x for x in cell.parent.parent.iterdir() if x.is_file() and x.stem==stem
                    and x.suffix.lower() in {'.bmp','.png','.jpg','.jpeg','.tif','.tiff'}]
        hashes=[]
        for field in candidates:
            with Image.open(field) as im:
                im=im.convert('RGB');hashes.append(hashlib.sha256(str(im.size).encode()+np.asarray(im).tobytes()).hexdigest())
        if len(set(hashes))>1:raise ValueError('Ambiguous source-field files for '+group)
        source_hashes[group]=hashes[0] if hashes else ''
    frame['source_image_sha256']=frame.source_group.map(source_hashes)
    # Union source-image groups connected by exact duplicate pixels, then deduplicate.
    parent={x:x for x in frame.source_group.unique()}
    def find(x):
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:parent[max(a,b)]=min(a,b)
    for _,sub in frame[frame.source_image_sha256!=''].groupby('source_image_sha256',sort=False):
        groups=list(sub.source_group.unique())
        for g in groups[1:]:union(groups[0],g)
    for _,sub in frame.groupby('pixel_sha256',sort=False):
        if sub.label.nunique()>1:raise ValueError('Identical pixels have conflicting labels; resolve the dataset before training.')
        groups=list(sub.source_group.unique())
        for g in groups[1:]:union(groups[0],g)
    frame['group_id']=frame.source_group.map(lambda x:find(x))
    duplicate_rows=frame[frame.duplicated('pixel_sha256',keep='first')].copy()
    frame=frame.drop_duplicates('pixel_sha256',keep='first').reset_index(drop=True)
    return frame,duplicate_rows

def augment_pil(im):
    if random.random()<.5: im=ImageOps.mirror(im)
    if random.random()<.5: im=ImageOps.flip(im)
    im=im.rotate(90*random.randrange(4),expand=True)
    im=ImageEnhance.Color(im).enhance(random.uniform(.92,1.08))
    im=ImageEnhance.Brightness(im).enhance(random.uniform(.94,1.06))
    return im

def letterbox(im,size):
    w,h=im.size; scale=min(size/w,size/h)
    nw,nh=max(1,round(w*scale)),max(1,round(h*scale)); left=(size-nw)//2; top=(size-nh)//2
    canvas=Image.new('RGB',(size,size),(255,255,255))
    canvas.paste(im.resize((nw,nh),Image.Resampling.BILINEAR),(left,top))
    rgb=torch.from_numpy(np.asarray(canvas).copy()).permute(2,0,1).float()/255.
    valid=torch.zeros(1,size,size); valid[:,top:top+nh,left:left+nw]=1.
    meta=(left,top,nw,nh,w,h)
    return rgb,valid,meta

def morphology_proxies(rgb,valid):
    # Image-derived proxies, not ground-truth segmentation or verified anatomy.
    gray=.299*rgb[:,0:1]+.587*rgb[:,1:2]+.114*rgb[:,2:3]
    sat=rgb.amax(1,keepdim=True)-rgb.amin(1,keepdim=True)
    nucleus=(1-gray).clamp_min(.01).pow(2)
    cyto=(sat+.03)*(gray+.10)
    dx=F.pad((gray[:,:,:,1:]-gray[:,:,:,:-1]).abs(),(0,1,0,0))
    dy=F.pad((gray[:,:,1:,:]-gray[:,:,:-1,:]).abs(),(0,0,0,1))
    interior=(F.avg_pool2d(valid,5,1,2)>.99).to(rgb.dtype)
    edge=(dx+dy+.005)*interior
    prior=torch.cat([nucleus,cyto,edge],1)*valid
    return prior.clamp_min(1e-8)

class Cells(Dataset):
    def __init__(self,frame,training=False):self.frame=frame.reset_index(drop=True);self.training=training
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(r.path) as im:im=im.convert('RGB')
        if self.training:im=augment_pil(im)
        rgb,valid,meta=letterbox(im,CFG.image_size)
        return {'rgb':rgb,'valid':valid,'raw':np.asarray(im).copy(),'meta':meta,'y':int(r.label),'id':str(r.sample_id)}

def collate(items):
    return {'rgb':torch.stack([x['rgb'] for x in items]),'valid':torch.stack([x['valid'] for x in items]),
            'y':torch.tensor([x['y'] for x in items]),'raw':[x['raw'] for x in items],
            'meta':[x['meta'] for x in items],'ids':[x['id'] for x in items]}

def loader(frame,training=False,batch_size=None):
    return DataLoader(Cells(frame,training),batch_size=batch_size or CFG.batch_size,shuffle=training,
                      num_workers=CFG.num_workers,collate_fn=collate,pin_memory=torch.cuda.is_available())

def normalize_teacher(x):return (x-MEAN.to(x))/STD.to(x)

## Original model definitions

The full global arm retains your original implementation:

**RGB → TinyViT-5M → 192D Conv/GN/SiLU projection → three-query, four-head custom attention → LayerNorm → dropout + five-class linear classifier.**

The three image-derived priors are darkness, colour/intensity and boundary change. They are weak cues, not verified nucleus/cytoplasm segmentation. The residual attention contribution is unchanged.

KD uses frozen ConvNeXt-Tiny logits with temperature 2 and the standard temperature-squared KL scaling. Both KD arms share the exact same teacher. Teacher parameters and compute are excluded from student inference measurements, because the teacher runs only during training.

In [7]:
class ConvGN(nn.Sequential):
    def __init__(self,a,b,k=1,s=1,groups=1,act=True):
        gn=math.gcd(8,b)
        super().__init__(nn.Conv2d(a,b,k,s,k//2,groups=groups,bias=False),nn.GroupNorm(gn,b),
                         nn.SiLU(inplace=False) if act else nn.Identity())

class QueryMorphologyAttention(nn.Module):
    """Three learned morphology queries, with image-derived proxy biases.

    Uses original encoder resolution. The scorer and four attention heads are learned;
    proxy maps are weak cues, not nucleus/cytoplasm ground-truth masks.
    """
    def __init__(self, c, heads):
        super().__init__()
        self.heads, self.c = heads, c
        self.score = nn.Sequential(nn.Conv2d(c, max(4,c//4), 1), nn.GELU(), nn.Conv2d(max(4,c//4),3,1))
        self.prior_strength = nn.Parameter(torch.zeros(3))
        self.queries = nn.Parameter(torch.randn(3,c)*.02)
        self.position = nn.Linear(2,c,bias=False)
        self.token_norm = nn.LayerNorm(c)
        self.query_norm = nn.LayerNorm(c)
        self.cross = nn.MultiheadAttention(c,heads,batch_first=True,dropout=0.)
        self.region_norm = nn.LayerNorm(c)
        self.channel_mix = nn.Linear(3*c,c)
    def forward(self, f, prior, valid):
        b,c,h,w = f.shape
        mask = F.adaptive_avg_pool2d(valid,(h,w)).flatten(1) > .05
        if not mask.any(1).all(): raise ValueError('No valid image tokens')
        yy,xx = torch.meshgrid(torch.linspace(-1,1,h,device=f.device,dtype=f.dtype),
                               torch.linspace(-1,1,w,device=f.device,dtype=f.dtype),indexing='ij')
        xy = torch.stack((xx,yy),-1).reshape(1,h*w,2)
        tokens = f.flatten(2).transpose(1,2)
        keys = self.token_norm(tokens+self.position(xy))
        p = F.adaptive_avg_pool2d(prior,(h,w)).clamp_min(1e-8)
        bias = (self.score(f)+F.softplus(self.prior_strength)[None,:,None,None]*p.log()).flatten(2)
        bias = bias.masked_fill(~mask[:,None,:],-1e4)
        bias = bias[:,None].expand(-1,self.heads,-1,-1).reshape(b*self.heads,3,h*w)
        queries = self.query_norm(self.queries)[None].expand(b,-1,-1)
        regions,weights = self.cross(queries,keys,tokens,attn_mask=bias,need_weights=True)
        pooled = (tokens*mask[:,:,None]).sum(1)/mask.sum(1,keepdim=True)
        z = pooled + .25*self.channel_mix(self.region_norm(regions).flatten(1))
        return z,weights.reshape(b,3,h,w)

class Student(nn.Module):
    def __init__(self,pretrained=False):
        super().__init__()
        self.backbone=timm.create_model(STUDENT_NAME,pretrained=pretrained,num_classes=0)
        self.backbone.head=nn.Identity()
        self.register_buffer('mean',torch.tensor(self.backbone.pretrained_cfg['mean']).view(1,3,1,1))
        self.register_buffer('std',torch.tensor(self.backbone.pretrained_cfg['std']).view(1,3,1,1))
        self.projection=ConvGN(self.backbone.num_features,CFG.feature_dim)
        self.attention=QueryMorphologyAttention(CFG.feature_dim,CFG.attention_heads)
        self.norm=nn.LayerNorm(CFG.feature_dim)
        self.head=nn.Sequential(nn.Dropout(.15),nn.Linear(CFG.feature_dim,5))
    def forward(self,rgb,valid):
        f=self.backbone.forward_features((rgb-self.mean.to(rgb))/self.std.to(rgb))
        if f.ndim!=4:raise ValueError('TinyViT must return NCHW features: '+str(f.shape))
        z,maps=self.attention(self.projection(f),morphology_proxies(rgb,valid),valid)
        z=self.norm(z);return z,self.head(z),maps

def new_teacher(pretrained):
    from torchvision.models import convnext_tiny,ConvNeXt_Tiny_Weights
    m=convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,5)
    return m

def kd_loss(student,teacher):
    t=CFG.kd_temperature
    return F.kl_div(F.log_softmax(student.float()/t,1),F.softmax(teacher.detach().float()/t,1),reduction='batchmean')*t*t

@torch.no_grad()
def image_validation(model,frame,device,is_teacher=False):
    model.eval();ys=[];zs=[]
    for b in loader(frame,batch_size=CFG.teacher_batch_size if is_teacher else CFG.batch_size):
        x,v=b['rgb'].to(device),b['valid'].to(device)
        z=model(normalize_teacher(x)) if is_teacher else model(x,v)[1]
        ys.extend(b['y'].tolist());zs.append(z.float().cpu().numpy())
    return metric_values(ys,np.concatenate(zs))

## Matched model and initialization

Removing attention leaves **the same 192D projection, masked global pooling, LayerNorm and classifier**. Therefore A is a controlled “TinyViT + shared head” baseline, not the unmodified timm stock classification head. These shared components must be described in the paper.

For each seed, a common initialization is saved once. Its backbone, projection, normalization and classifier tensors are copied into all four arms and their hashes verified. A/B contain no unused custom-attention parameters. All arms receive the same epoch shuffle/augmentation schedule and 45-epoch learning-rate budget. Different seeds change the initialization and training RNG, not the split.

In [8]:
class AblationStudent(Student):
    def __init__(self, attention=True, pretrained=False):
        # Construct in the exact original order so all common tensors have the
        # same initialization for a given seed, including the final classifier.
        super().__init__(pretrained)
        self.use_attention = bool(attention)
        if not self.use_attention:
            del self.attention  # No unused attention parameters in the baseline.

    def forward(self, rgb, valid):
        f = self.backbone.forward_features((rgb-self.mean.to(rgb))/self.std.to(rgb))
        if f.ndim != 4:
            raise ValueError('Expected NCHW TinyViT features.')
        f = self.projection(f)
        if self.use_attention:
            z, maps = self.attention(f, morphology_proxies(rgb, valid), valid)
        else:
            mask = F.adaptive_avg_pool2d(valid, f.shape[-2:]).flatten(1) > .05
            if not mask.any(1).all():
                raise ValueError('No valid image tokens.')
            tokens = f.flatten(2).transpose(1, 2)
            z = (tokens*mask[:, :, None]).sum(1)/mask.sum(1, keepdim=True)
            maps = f.new_empty((len(f), 0, f.shape[-2], f.shape[-1]))
        z = self.norm(z)
        return z, self.head(z), maps


def state_digest(state, common_only=False):
    h = hashlib.sha256()
    for key, value in sorted(state.items()):
        if common_only and key.startswith('attention.'):
            continue
        a = value.detach().cpu().contiguous()
        h.update(key.encode()); h.update(str(a.dtype).encode()); h.update(str(tuple(a.shape)).encode())
        h.update(a.numpy().tobytes())
    return h.hexdigest()


def initialized_student(ctx, seed, arm):
    out = ctx['root']/'initialization'
    out.mkdir(exist_ok=True)
    path = out/f'seed_{seed}.pt'
    if not path.exists():
        seed_all(seed)
        model = Student(CFG.pretrained)
        state = cpu_state(model)
        save_pt({'model': state, 'seed': seed, 'contract': ctx['contract_hash'],
                 'common_sha256': state_digest(state, True)}, path)
        del model
    ck = load_pt(path)
    if ck['contract'] != ctx['contract_hash'] or ck['seed'] != seed:
        raise ValueError('Initialization contract mismatch.')
    model = AblationStudent(ARMS[arm]['attention'], pretrained=False)
    state = ck['model'] if ARMS[arm]['attention'] else {
        k: v for k, v in ck['model'].items() if not k.startswith('attention.')}
    install_state(model, state)
    if state_digest(cpu_state(model), True) != ck['common_sha256']:
        raise AssertionError('Common initialization differs across arms.')
    save_json({'seed': seed, 'arm': arm, 'common_sha256': ck['common_sha256'],
               'attention': ARMS[arm]['attention'], 'kd': ARMS[arm]['kd'],
               'pretraining': STUDENT_NAME, 'contract': ctx['contract_hash']},
              out/f'{arm}_seed_{seed}.json')
    return model

## Fixed split, paths and immutable experiment contract

Audits the real files, restores only compatible ablation outputs and records the code/data/runtime/config identity. Leave the frozen split unchanged. Reports-only outputs cannot resume model training.

In [9]:
def frozen_partition(scanned):
    mapping = pd.read_csv(io.StringIO(FROZEN_GROUP_SPLITS_CSV)).set_index('group_id')['split']
    frame = scanned.copy()
    if set(frame.group_id) != set(mapping.index):
        raise ValueError('Source groups differ from the supplied 4049-cell study. '
                         'Use the original SIPaKMeD image dataset. No replacement split is created.')
    frame['split'] = frame.group_id.map(mapping)
    if object_hash(frame.to_dict('records')) != ORIGINAL_EXPERIMENT['dataset_sha256']:
        raise ValueError('Image paths, pixels or source fields differ from the original audited manifest. '
                         'Attach the original dataset; do not silently regenerate the split.')
    return frame


def assert_partitions(frame):
    if frame.sample_id.duplicated().any() or frame.relative_path.duplicated().any():
        raise ValueError('Duplicate sample identifiers.')
    if set(frame.split) != {'train', 'val', 'test'}:
        raise ValueError('Expected train, val and test partitions.')
    for column in ('group_id', 'pixel_sha256'):
        if frame.groupby(column).split.nunique().max() != 1:
            raise ValueError('Partition overlap: '+column)
    for split in ('train', 'val', 'test'):
        if set(frame.loc[frame.split == split, 'label']) != set(range(5)):
            raise ValueError('Every partition must contain all five classes.')


def restore_ablation():
    root = Path(CFG.output_root).expanduser()
    if CFG.resume_source:
        source = Path(CFG.resume_source).expanduser()
        if not (source/'ablation_contract.json').is_file():
            raise FileNotFoundError('resume_source must be the FULL output of THIS ablation notebook, '
                                    'containing ablation_contract.json and model .pt files.')
        if next(source.rglob('*.pt'), None) is None:
            raise FileNotFoundError('This folder contains reports but no model .pt files. '
                                    'Attach the FULL ablation output to resume; the reports ZIP cannot resume training.')
        if source.resolve() != root.resolve():
            if root.exists() and any(root.iterdir()):
                print('Existing working ablation output takes priority; its contract will be checked.')
            else:
                shutil.copytree(source, root, dirs_exist_ok=True)
    root.mkdir(parents=True, exist_ok=True)
    return root


def runtime_identity():
    return {'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
            'timm': timm.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
            'sklearn': sklearn.__version__, 'python': platform.python_version()}


def prepare_ablation():
    if CFG.image_size != 224 or CFG.feature_dim != 192 or CFG.attention_heads != 4:
        raise ValueError('This is a fixed-architecture 224px, 192D, 4-head ablation.')
    if tuple(CFG.seeds) != (42, 123, 2026):
        raise ValueError('The final ablation is predeclared for seeds (42, 123, 2026).')
    if min(CFG.teacher_epochs, CFG.student_epochs, CFG.bootstrap_replicates,
           CFG.benchmark_images, CFG.benchmark_repeats) < 1:
        raise ValueError('Budgets must be positive.')
    if CFG.num_workers != 0:
        raise ValueError('Keep num_workers=0 to preserve the original augmentation/shuffle schedule.')
    root = restore_ablation()
    dataset = resolve_dataset(CFG.dataset_root)
    scanned, duplicates = scan_cells(dataset)
    frame = frozen_partition(scanned)
    assert_partitions(frame)
    scientific = asdict(CFG)
    for key in ('dataset_root', 'output_root', 'resume_source', 'fresh_source', 'reuse_verified_fresh',
                'allow_cpu', 'session_hours', 'run_final_test', 'benchmark_images',
                'benchmark_repeats', 'benchmark_warmup', 'bootstrap_replicates', 'bootstrap_seed'):
        scientific.pop(key)
    contract = {'version': VERSION, 'implementation_sha256': implementation_hash(),
                'config': scientific, 'arms': ARMS, 'primary_arm': PRIMARY_ARM,
                'classes': list(CLASSES), 'dataset_sha256': object_hash(frame.to_dict('records')),
                'runtime': runtime_identity(), 'student': STUDENT_NAME,
                'selection': 'Within each run: validation macro-F1, then accuracy, then lower CE',
                'aggregation': 'Mean and sample SD over three single-model seeds; no ensemble',
                'scope': 'Post-selection follow-up: this dataset and test partition have previously been examined',
                'grouping': 'Source-image groups with exact-duplicate union; patient identities unverified',
                'shared_teacher': 'One frozen ConvNeXt-Tiny checkpoint for all KD arms and seeds'}
    ch = object_hash(contract); contract['contract_hash'] = ch
    path = root/'ablation_contract.json'
    if path.exists() and read_json(path) != contract:
        raise ValueError('Ablation code, data, settings or runtime changed. Restore the original environment/settings '
                         'or use a separate output_root. Existing checkpoints are not overwritten.')
    save_json(contract, path)
    frame.to_csv(root/'manifest.csv', index=False)
    duplicates.to_csv(root/'duplicates_removed.csv', index=False)
    frame.groupby(['split', 'class_name']).agg(images=('sample_id','size'), groups=('group_id','nunique')).reset_index().to_csv(root/'split_summary.csv', index=False)
    save_json({'images': len(frame), 'split_counts': frame.split.value_counts().to_dict(),
               'source_groups_disjoint': True, 'patient_grouping_verified': False,
               'exact_duplicates_removed': len(duplicates), 'near_duplicate_detection': False,
               'split': 'Exact original saved manifest, not a newly sampled split',
               'test_access_at_setup': 'File/pixel hashing for integrity only; no fitting or checkpoint selection'},
              root/'data_audit.json')
    frame['path'] = frame.relative_path.map(lambda p: str(dataset/p))
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cpu' and not CFG.allow_cpu:
        raise RuntimeError('Enable a Kaggle GPU. The notebook intentionally uses one GPU for all arms.')
    ctx = {'root': root, 'dataset': dataset, 'frame': frame, 'device': device, 'contract_hash': ch,
           'contract': contract, 'deadline': time.monotonic()+CFG.session_hours*3600}
    for split in ('train', 'val', 'test'):
        ctx[split] = frame[frame.split == split].reset_index(drop=True)
    print('Exact split:', frame.split.value_counts().to_dict(), '\nDevice:', device,
          '\nStudent jobs:', len(ARMS)*len(CFG.seeds), flush=True)
    return ctx

## Verified reuse of the completed fresh experiment

A previous teacher is adopted only when its experiment record and checkpoint SHA-256 match the supplied report and the training recipe matches. If no eligible teacher weights are attached, the teacher is trained once with the original 12-epoch recipe.

The historical student is not reused: its initial tensors were not saved. Retraining D with the other arms gives an auditable paired comparison. This is training-budget matching, not equal wall-clock training cost: KD legitimately adds teacher forward passes.

In [10]:
def find_fresh_source(ctx):
    if not CFG.reuse_verified_fresh:
        return None
    if CFG.fresh_source:
        hint = Path(CFG.fresh_source).expanduser()
        if not hint.is_dir():
            raise FileNotFoundError('fresh_source must be an extracted FULL output folder: '+str(hint))
        candidates = [hint] if (hint/'experiment.json').is_file() else [p.parent for p in hint.rglob('experiment.json')]
    else:
        candidates = []
        for base in (Path('/kaggle/input'), Path('/kaggle/working')):
            if base.exists():
                candidates.extend(p.parent for p in base.rglob('experiment.json'))
    verified, rejected = [], []
    for source in sorted(set(candidates)):
        try:
            exp = read_json(source/'experiment.json')
            if exp != ORIGINAL_EXPERIMENT:
                continue
            tpath = source/'teacher/best.pt'
            if not tpath.is_file():
                rejected.append({'path': str(source), 'reason': 'Reports only: teacher/best.pt is missing'})
                continue
            if file_hash(tpath) != ORIGINAL_TEACHER_SHA:
                rejected.append({'path': str(source), 'reason': 'Teacher bytes differ from the reported run'})
                continue
            verified.append(source)
        except (OSError, ValueError, TypeError) as exc:
            rejected.append({'path': str(source), 'reason': str(exc)})
    # All eligible candidates have the same verified teacher. Prefer one with
    # the exact student too; duplicated Kaggle mounts are not a reason to fail.
    verified.sort(key=lambda p: (not (p/'seed_42/student/best.pt').exists(), len(str(p)), str(p)))
    source = verified[0] if verified else None
    save_json({'selected': str(source) if source else None, 'rejected': rejected,
               'fallback': 'Train only missing jobs using the fixed recipe if no compatible weights are available'},
              ctx['root']/'fresh_source_resolution.json')
    print('Verified fresh source:', source if source else 'none; missing stages will train', flush=True)
    return source


def original_recipe_matches():
    keys = ('image_size','feature_dim','attention_heads','batch_size','teacher_batch_size','num_workers',
            'teacher_seed','teacher_epochs','student_epochs','teacher_lr','student_lr','weight_decay',
            'label_smoothing','kd_weight','kd_temperature','pretrained','amp')
    return all(getattr(CFG, k) == ORIGINAL_EXPERIMENT['config'][k] for k in keys)


def adopt_best(ctx, name, source_path, expected_sha, epochs, expected_epoch=None):
    out = ctx['root']/name
    if (out/'complete.json').exists() or (out/'last.pt').exists():
        return False
    if not source_path.is_file() or file_hash(source_path) != expected_sha:
        return False
    ck = load_pt(source_path)
    if ck.get('contract') != ORIGINAL_EXPERIMENT['contract_hash']:
        raise ValueError('Original checkpoint contract mismatch.')
    if expected_epoch is not None and ck['epoch'] != expected_epoch:
        raise ValueError('Original best epoch does not match the report.')
    out.mkdir(parents=True, exist_ok=True)
    save_pt({'model': ck['model'], 'epoch': ck['epoch'], 'metrics': ck['metrics'],
             'contract': ctx['contract_hash'], 'adopted_from_sha256': expected_sha}, out/'best.pt')
    for filename in ('history.csv',):
        p = source_path.parent/filename
        if p.is_file():
            shutil.copy2(p, out/filename)
    save_json({'contract': ctx['contract_hash'], 'epochs': epochs, 'best_epoch': ck['epoch'],
               'best_sha256': file_hash(out/'best.pt'), 'adopted': True,
               'source_sha256': expected_sha, 'source': str(source_path),
               'scope': 'Existing completed training reused; not an additional independent replication'}, out/'complete.json')
    print('Reused verified completed stage:', name, flush=True)
    return True


def reuse_original(ctx, source):
    if source is None or not original_recipe_matches():
        return
    adopt_best(ctx, 'teacher', source/'teacher/best.pt', ORIGINAL_TEACHER_SHA, CFG.teacher_epochs, 12)
    # The original fresh study did not preserve initial student tensor hashes.
    # Rerun all four arms for every seed from the audited common initialization.
    save_json({'teacher_reuse': 'Exact reported teacher bytes can be adopted',
               'historical_student_reuse': False,
               'reason': 'No saved original initialization snapshot; the primary paired ablation requires audited shared initial tensors',
               'resume': 'Completed student jobs from THIS ablation remain reusable'},
              ctx['root']/'historical_reuse_policy.json')

## Training, checkpoints and epoch-boundary resume

Every job saves `best.pt`, `last.pt`, `history.csv` and `complete.json`. Best means **validation macro-F1**, then accuracy, then lower cross-entropy. The complete 45-epoch budget is used for every student; no test-based early stopping or best-test-seed selection occurs.

Losses use label smoothing 0.05, AdamW, weight decay 1e-4, gradient clipping at norm 2 and the original cosine schedule. A/C do not call the teacher. B/D use CE + 0.25 KD. `num_workers=0` preserves the original paired augmentation schedule.

The teacher is one fixed model shared across the three student seeds; seed variability is conditional on that teacher and does not measure teacher-training variability.

In [11]:
def completed_stage(ctx, name, epochs):
    out = ctx['root']/name
    p = out/'complete.json'
    if not p.exists():
        return False
    receipt = read_json(p)
    best = out/'best.pt'
    if (receipt['contract'] != ctx['contract_hash'] or receipt['epochs'] != epochs
            or not best.exists() or receipt['best_sha256'] != file_hash(best)):
        raise ValueError('Saved stage is inconsistent: '+name)
    return True


def fit_job(ctx, name, model, epochs, lr, seed, train_epoch, validate):
    out = ctx['root']/name; out.mkdir(parents=True, exist_ok=True)
    if completed_stage(ctx, name, epochs):
        install_state(model, load_pt(out/'best.pt')['model'])
        return model
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.weight_decay)
    use_amp = CFG.amp and ctx['device'].type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    start, history, best_key, best_epoch = 0, [], (-1.,), 0
    last, best = out/'last.pt', out/'best.pt'
    if last.exists():
        ck = load_pt(last)
        if ck['contract'] != ctx['contract_hash']:
            raise ValueError('Resume contract mismatch: '+name)
        install_state(model, ck['model']); opt.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        start = int(ck['epoch']); history = ck['history']
        best_key = tuple(ck['best_key']); best_epoch = int(ck['best_epoch'])
        if not 0 <= start <= epochs:
            raise ValueError('Resume epoch outside the fixed training budget.')
    for epoch in range(start+1, epochs+1):
        budget(ctx); seed_all(seed+epoch*1009)
        epoch_lr = lr*(.1+.9*.5*(1+math.cos(math.pi*(epoch-1)/max(1,epochs))))
        for group in opt.param_groups:
            group['lr'] = epoch_lr
        started = time.monotonic()
        train_info = train_epoch(model, opt, scaler, epoch)
        model.eval(); clear_bias_cache(model)
        metrics = validate(model)
        row = {'epoch': epoch, 'lr': epoch_lr, **train_info, **metrics,
               'seconds': time.monotonic()-started}
        history.append(row)
        key = quality(metrics)
        if key > best_key:
            best_key, best_epoch = key, epoch
            save_pt({'model': cpu_state(model), 'epoch': epoch, 'metrics': metrics,
                     'contract': ctx['contract_hash']}, best)
        save_pt({'model': cpu_state(model), 'optimizer': opt.state_dict(), 'scaler': scaler.state_dict(),
                 'epoch': epoch, 'history': history, 'best_key': list(best_key), 'best_epoch': best_epoch,
                 'contract': ctx['contract_hash']}, last)
        pd.DataFrame(history).to_csv(out/'history.csv', index=False)
        save_json({'status': 'training', 'stage': name, 'epoch': epoch, 'epochs': epochs,
                   'completed_student_jobs': count_complete(ctx), 'planned_student_jobs': 12},
                  ctx['root']/'session_status.json')
        print(f'{name} {epoch}/{epochs}: val acc {metrics["accuracy"]:.2%}, F1 {metrics["macro_f1"]:.2%}; '
              f'{row["seconds"]:.1f}s, stage ETA ~{row["seconds"]*(epochs-epoch)/60:.1f} min', flush=True)
    if not best.exists():
        raise ValueError('No valid best checkpoint: '+name)
    save_json({'contract': ctx['contract_hash'], 'epochs': epochs, 'best_epoch': best_epoch,
               'best_sha256': file_hash(best), 'adopted': False}, out/'complete.json')
    install_state(model, load_pt(best)['model'])
    return model


def train_epoch_function(ctx, teacher=None, is_teacher=False, use_kd=False):
    device = ctx['device']; use_amp = CFG.amp and device.type == 'cuda'
    if use_kd and teacher is None:
        raise ValueError('KD requires the frozen shared teacher.')
    def step(model, opt, scaler, epoch):
        model.train(); clear_bias_cache(model)
        total = ce_sum = kd_sum = 0.; n = 0
        # Same epoch RNG seed + num_workers=0 => paired order and augmentations.
        for b in loader(ctx['train'], True, CFG.teacher_batch_size if is_teacher else CFG.batch_size):
            x, v, y = b['rgb'].to(device), b['valid'].to(device), b['y'].to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(normalize_teacher(x)) if is_teacher else model(x, v)[1]
                kd = logits.new_zeros(())
                if use_kd:
                    with torch.no_grad():
                        tz = torch.cat([teacher(normalize_teacher(c)) for c in x.split(CFG.teacher_batch_size)])
                    kd = kd_loss(logits, tz)
                ce = F.cross_entropy(logits.float(), y, label_smoothing=CFG.label_smoothing)
                loss = ce + (CFG.kd_weight*kd if use_kd else 0.)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite loss.')
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.)
            scaler.step(opt); scaler.update()
            count = len(y); n += count
            total += float(loss.detach())*count; ce_sum += float(ce.detach())*count
            kd_sum += float(kd.detach())*count
        return {'train_loss': total/n, 'train_ce': ce_sum/n, 'train_kd': kd_sum/n}
    return step


def count_complete(ctx):
    return sum((ctx['root']/arm/f'seed_{seed}'/'complete.json').is_file()
               for arm in ARMS for seed in CFG.seeds)


def load_student(ctx, arm, seed):
    if not completed_stage(ctx, f'{arm}/seed_{seed}', CFG.student_epochs):
        raise ValueError('Student training is not complete.')
    model = AblationStudent(ARMS[arm]['attention'], False)
    ck = load_pt(ctx['root']/arm/f'seed_{seed}'/'best.pt')
    if ck['contract'] != ctx['contract_hash']:
        raise ValueError('Student checkpoint identity mismatch.')
    install_state(model, ck['model'])
    return freeze(model.to(ctx['device']))

## Per-run predictions and the pre-test lock

All twelve validation evaluations must finish before `evaluation_lock.json` is written. It records the selected checkpoint hashes. **D_FULL remains the predeclared primary model regardless of test results.** All four arms are evaluated for comparison; no arm or seed is chosen by its test score.

Each run gets metrics, per-image probabilities, per-class precision/recall/F1 and confusion matrices. Saved evaluations are reused only if their signatures and report hashes match. Means are over three individual models evaluated on the same 762 images; they are not ensemble predictions.

In [12]:
@torch.no_grad()
def predict_frame(ctx, model, split):
    if split == 'test' and not (ctx['root']/'evaluation_lock.json').exists():
        raise RuntimeError('All 12 validation results must be locked before test inference.')
    model.eval(); clear_bias_cache(model)
    ids, logits = [], []
    for b in loader(ctx[split]):
        x, v = b['rgb'].to(ctx['device']), b['valid'].to(ctx['device'])
        ids.extend(b['ids']); logits.append(model(x, v)[1].float().cpu().numpy())
    if ids != ctx[split].sample_id.astype(str).tolist():
        raise AssertionError('Prediction order does not match the fixed manifest.')
    return np.concatenate(logits)


def write_prediction_report(frame, logits, out):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    y = frame.label.to_numpy(); p = torch.tensor(logits).softmax(1).numpy(); pred = p.argmax(1)
    metrics = metric_values(y, logits)
    table = frame[['sample_id','relative_path','group_id','class_name','label']].copy()
    table['prediction'] = pred; table['predicted_class'] = [CLASSES[i] for i in pred]
    table['confidence'] = p.max(1)
    for i, cls in enumerate(CLASSES):
        table['p_'+cls] = p[:, i]
    table.to_csv(out/'predictions.csv', index=False)
    save_json(metrics, out/'metrics.json')
    report = classification_report(y, pred, labels=list(range(5)), target_names=CLASSES,
                                   output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(out/'classification_report.csv')
    cm = confusion_matrix(y, pred, labels=list(range(5)))
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(out/'confusion_matrix.csv')
    fig, ax = plt.subplots(figsize=(7,6)); ax.imshow(cm, cmap='Blues')
    ax.set(xticks=range(5), yticks=range(5), xticklabels=CLASSES, yticklabels=CLASSES,
           xlabel='Predicted', ylabel='True', title=f'Accuracy {metrics["accuracy"]:.2%} | Macro-F1 {metrics["macro_f1"]:.2%}')
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    for i in range(5):
        for j in range(5):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j] > cm.max()/2 else 'black')
    fig.tight_layout(); fig.savefig(out/'confusion_matrix.png', dpi=140); plt.close(fig)
    return metrics


def evaluate_run(ctx, model, arm, seed, split, lock_hash=None):
    root = ctx['root']/('final_test' if split == 'test' else 'validation')/arm/f'seed_{seed}'
    signature = {'contract': ctx['contract_hash'], 'checkpoint_sha256': file_hash(ctx['root']/arm/f'seed_{seed}'/'best.pt'),
                 'split': split, 'ids': object_hash(ctx[split].sample_id.tolist()), 'lock_hash': lock_hash}
    receipt = root/'complete.json'
    if receipt.exists():
        saved = read_json(receipt)
        if saved['signature'] != signature:
            raise ValueError('Saved evaluation identity differs: '+str(root))
        for name, expected in saved['report_sha256'].items():
            if not (root/name).is_file() or file_hash(root/name) != expected:
                raise ValueError('Saved evaluation report changed: '+str(root/name))
        metrics = read_json(root/'metrics.json')
    else:
        budget(ctx)
        logits = predict_frame(ctx, model, split)
        metrics = write_prediction_report(ctx[split], logits, root)
        save_json({'signature': signature, 'report_sha256': {
            name: file_hash(root/name) for name in ('predictions.csv','metrics.json','classification_report.csv','confusion_matrix.csv')}}, receipt)
    return {'arm': arm, 'seed': seed, 'split': split, **metrics}


def results_summary(rows, path):
    table = pd.DataFrame(rows); table.to_csv(path, index=False)
    summary = table.groupby('arm', sort=False).agg(
        seeds=('seed','nunique'), n_unique_images=('n','first'),
        accuracy_mean=('accuracy','mean'), accuracy_seed_sd=('accuracy',lambda x: float(x.std(ddof=1)) if len(x)>1 else np.nan),
        macro_f1_mean=('macro_f1','mean'), macro_f1_seed_sd=('macro_f1',lambda x: float(x.std(ddof=1)) if len(x)>1 else np.nan),
        balanced_accuracy_mean=('balanced_accuracy','mean')).reset_index()
    summary['aggregation'] = 'Three single-model runs on the same images; not an ensemble'
    summary.to_csv(Path(path).with_name(Path(path).stem+'_summary.csv'), index=False)
    return summary


def lock_evaluation(ctx, validation_rows):
    if len(validation_rows) != len(ARMS)*len(CFG.seeds):
        raise ValueError('All twelve validation evaluations are required.')
    checkpoints = {}
    for arm in ARMS:
        for seed in CFG.seeds:
            name = f'{arm}/seed_{seed}'
            if not completed_stage(ctx, name, CFG.student_epochs):
                raise ValueError('Training is incomplete: '+name)
            p = ctx['root']/name/'best.pt'; checkpoints[str(p.relative_to(ctx['root']))] = file_hash(p)
    lock = {'contract': ctx['contract_hash'], 'primary_arm': PRIMARY_ARM, 'arms': list(ARMS),
            'seeds': list(CFG.seeds), 'checkpoint_sha256': checkpoints,
            'checkpoint_selection': 'Validation macro-F1, then accuracy, then lower CE',
            'primary_arm_selection': 'Predeclared full global model; no test-based arm or seed selection',
            'scope': 'Post-selection follow-up on the previously inspected source-group test partition'}
    path = ctx['root']/'evaluation_lock.json'
    if path.exists() and read_json(path) != lock:
        raise ValueError('Existing evaluation lock changed. No test overwrite is allowed.')
    save_json(lock, path)
    return lock

## Source-group bootstrap and paired factorial effects

Two different uncertainties are reported:

| Report | Meaning |
|---|---|
| Seed SD | Sample SD of accuracy/F1 across the three trained student seeds |
| Source-group 95% CI | Resample source groups with replacement, keeping cells within a group together |

The same bootstrap group draws are shared across all arms and seeds. Per-seed CIs and CIs for the mean of the **three fixed fitted models** are saved separately. Seeds are not pooled as extra independent test images; patients are not assumed to be known.

Paired contrasts include C−A, D−B, B−A, D−C, D−A and the interaction D−C−B+A. Differences are in **percentage points**. These exploratory intervals are not adjusted for multiple comparisons or prior test inspection. They cannot remove the post-selection limitation.

In [13]:
def confusion_metrics(cm):
    cm = np.asarray(cm, dtype=np.float64)
    diag = np.diagonal(cm, axis1=-2, axis2=-1)
    actual, predicted = cm.sum(-1), cm.sum(-2)
    accuracy = diag.sum(-1)/cm.sum(axis=(-2,-1))
    f1 = np.divide(2*diag, actual+predicted, out=np.zeros_like(diag), where=(actual+predicted)>0).mean(-1)
    return np.stack([accuracy, f1], axis=-1)


def test_group_confusions(ctx):
    frame = ctx['test']; group_ids = sorted(frame.group_id.unique()); gmap = {g:i for i,g in enumerate(group_ids)}
    cm = np.zeros((len(group_ids), len(CFG.seeds), len(ARMS), 5, 5), dtype=np.int64)
    for ai, arm in enumerate(ARMS):
        for si, seed in enumerate(CFG.seeds):
            path = ctx['root']/'final_test'/arm/f'seed_{seed}'/'predictions.csv'
            pred = pd.read_csv(path)
            if (pred.sample_id.tolist() != frame.sample_id.tolist()
                    or not np.array_equal(pred.label, frame.label)
                    or pred.group_id.tolist() != frame.group_id.tolist()):
                raise ValueError('Bootstrap prediction alignment failed: '+str(path))
            np.add.at(cm, ([gmap[g] for g in pred.group_id], si, ai, pred.label, pred.prediction), 1)
    return cm, group_ids


def bootstrap_reports(ctx):
    cm, groups = test_group_confusions(ctx)
    rng = np.random.default_rng(CFG.bootstrap_seed)
    # Cluster resampling preserves all crops from each sampled source group.
    # The SAME group draws are used for every arm and every seed (paired design).
    weights = rng.multinomial(len(groups), np.full(len(groups), 1/len(groups)), size=CFG.bootstrap_replicates)
    sampled = np.einsum('bg,gsaij->bsaij', weights, cm, optimize=True)
    valid = (sampled.sum(-1)>0).all(axis=(-1,-2,-3))
    if valid.sum() < max(20, int(.9*CFG.bootstrap_replicates)):
        raise ValueError('Too many bootstrap draws omit a class; review group support.')
    boot = confusion_metrics(sampled[valid])  # replicates x seeds x arms x metrics
    point = confusion_metrics(cm.sum(0))      # seeds x arms x metrics
    metric_names = ('accuracy','macro_f1'); arms = list(ARMS)
    rows = []
    for ai, arm in enumerate(arms):
        for mi, metric in enumerate(metric_names):
            for si, seed in enumerate(CFG.seeds):
                lo, hi = np.quantile(boot[:,si,ai,mi], [.025,.975])
                rows.append({'arm': arm, 'seed': seed, 'metric': metric, 'estimate': point[si,ai,mi],
                             'ci95_low': lo, 'ci95_high': hi, 'scope': 'Fixed model; source-group bootstrap'})
            lo, hi = np.quantile(boot[:,:,ai,mi].mean(1), [.025,.975])
            rows.append({'arm': arm, 'seed': 'mean_of_3', 'metric': metric,
                         'estimate': point[:,ai,mi].mean(), 'ci95_low': lo, 'ci95_high': hi,
                         'scope': 'Mean of the three fixed seeds; source-group uncertainty, not seed uncertainty'})
    pd.DataFrame(rows).to_csv(ctx['root']/'final_test/test_group_bootstrap_ci.csv', index=False)
    # Ordered factorial contrasts, with D-A added as the overall comparison.
    contrasts = {'attention_without_KD_C_minus_A': [-1,0,1,0],
                 'attention_with_KD_D_minus_B': [0,-1,0,1],
                 'KD_without_attention_B_minus_A': [-1,1,0,0],
                 'KD_with_attention_D_minus_C': [0,0,-1,1],
                 'full_minus_plain_D_minus_A': [-1,0,0,1],
                 'interaction_D_minus_C_minus_B_plus_A': [1,-1,-1,1]}
    effects, paired = [], []
    for label, coefficients in contrasts.items():
        c = np.asarray(coefficients)
        pe = np.einsum('sam,a->sm', point, c)
        be = np.einsum('bsam,a->bsm', boot, c).mean(1)
        for mi, metric in enumerate(metric_names):
            lo, hi = np.quantile(be[:,mi], [.025,.975])
            effects.append({'contrast': label, 'metric': metric, 'mean_difference_pp': 100*pe[:,mi].mean(),
                            'seed_sd_pp': 100*pe[:,mi].std(ddof=1), 'ci95_low_pp': 100*lo,
                            'ci95_high_pp': 100*hi, 'positive_seed_count': int((pe[:,mi]>0).sum()),
                            'scope': 'Exploratory paired cluster CI; fixed seeds; not multiplicity-adjusted'})
            for si, seed in enumerate(CFG.seeds):
                paired.append({'contrast': label, 'seed': seed, 'metric': metric, 'difference_pp': 100*pe[si,mi]})
    pd.DataFrame(effects).to_csv(ctx['root']/'final_test/paired_effects.csv', index=False)
    pd.DataFrame(paired).to_csv(ctx['root']/'final_test/paired_effects_by_seed.csv', index=False)
    save_json({'replicates_requested': CFG.bootstrap_replicates, 'replicates_valid': int(valid.sum()),
               'seed': CFG.bootstrap_seed, 'unique_test_images': len(ctx['test']), 'source_groups': len(groups),
               'resampling_unit': 'source group, NOT individual cell and NOT patient',
               'seed_sd': 'Sample SD over 3 independently initialized student runs, conditional on one shared teacher',
               'ci': 'Percentile 95% cluster bootstrap with paired draws; conditional on fitted models',
               'not': 'Not an ensemble, not cross-validation, not external validation, not 3x as many independent images',
               'selection_bias': 'Does not correct previous dataset/test inspection or model-development selection'},
              ctx['root']/'final_test/uncertainty_protocol.json')

## Deployment weights and compute accounting

Every arm/seed exports a student-only checkpoint. The efficiency phase measures **all twelve students in one invocation on the same device**, with randomized fixed run order, warm-up and identical validation images.

| Quantity | Definition |
|---|---|
| Parameters | Actual parameters in the deployed student |
| Checkpoint MB | Actual serialized student bundle size |
| Matrix GMACs | Counted convolution, linear and attention matrix multiply-accumulates |
| Matrix FLOPs | Twice the counted matrix MACs, under the multiply-plus-add convention |
| fvcore GOPs | Its counted operations under the one-FMA-is-one convention |
| Forward latency | Batch 1, FP32 tensors, resident input tensors; includes active normalization/cues/attention |
| Decoded-image-to-logits latency | Adds letterboxing and CPU-to-device transfer; starts from decoded PIL images |

Disk decoding is excluded from both timing measurements. CUDA timings are synchronized; hardware/runtime/TF32 flags and raw latency samples are saved. The teacher, PPO, crop selection and TTA are absent at inference.

Custom counters cover fused scaled-dot-product attention and biased batch matrix multiplication. Remaining unsupported elementwise/reduction operators are listed in each compute JSON. **Do not present the estimate as an exact all-operation FLOP count.** If matrix counting fails, the report says `failed` rather than silently reporting zero.

Resuming this phase measures all arms again to avoid mixing GPU types or sessions in the final timing table.

In [14]:
def export_student(ctx, model, arm, seed):
    out = ctx['root']/'deployment'/arm/f'seed_{seed}'; out.mkdir(parents=True, exist_ok=True)
    path = out/'model.pt'
    save_pt({'model': cpu_state(model), 'arm': arm, 'classes': list(CLASSES), 'version': VERSION,
             'student_name': STUDENT_NAME, 'config': {'image_size': CFG.image_size,
             'feature_dim': CFG.feature_dim, 'attention_heads': CFG.attention_heads},
             'contract': ctx['contract_hash']}, path)
    return path


class LogitWrapper(nn.Module):
    def __init__(self, model):
        super().__init__(); self.model = model
    def forward(self, rgb, valid):
        return self.model(rgb, valid)[1]


def compute_accounting(model, rgb, valid):
    # fvcore counts a multiply-add as ONE operation. We report that convention
    # explicitly, retain every unsupported operator, and do not call it exact FLOPs.
    clear_bias_cache(model)
    was_fast = torch.backends.mha.get_fastpath_enabled()
    try:
        from fvcore.nn.jit_handles import get_shape
        def sdpa_macs(inputs, outputs):
            q, k, v = [get_shape(x) for x in inputs[:3]]
            if any(shape is None or any(d is None for d in shape) for shape in (q,k,v)):
                raise ValueError('Unknown attention tensor shape during compute accounting.')
            batches = math.prod(q[:-2]); length, width = q[-2:]; keys = k[-2]
            return {'attention_matmul': batches*length*keys*(width+v[-1])}
        def baddbmm_macs(inputs, outputs):
            a, b = get_shape(inputs[1]), get_shape(inputs[2])
            return {'bmm': math.prod(a[:-2])*a[-2]*a[-1]*b[-1]}
        torch.backends.mha.set_fastpath_enabled(False)
        with torch.no_grad(), warnings.catch_warnings():
            warnings.simplefilter('ignore', torch.jit.TracerWarning)
            analysis = FlopCountAnalysis(LogitWrapper(model), (rgb, valid))
            analysis.set_op_handle('aten::scaled_dot_product_attention', sdpa_macs,
                                   'aten::baddbmm', baddbmm_macs)
            analysis.unsupported_ops_warnings(False); analysis.uncalled_modules_warnings(False)
            total = int(analysis.total())
            unsupported = dict(analysis.unsupported_ops())
            operators = {k: int(v) for k,v in analysis.by_operator().items()}
        missing_matrix = [op for op in unsupported if any(term in op for term in ('attention','mm','conv','linear','einsum'))]
        if missing_matrix:
            raise ValueError('Uncounted matrix/attention operations: '+str(missing_matrix))
        matrix_macs = sum(v for k,v in operators.items() if k in {'conv','linear','matmul','bmm','einsum','attention_matmul'})
        return {'status': 'measured', 'fvcore_counted_gops_1_fma': total/1e9,
                'counted_matrix_gmacs': matrix_macs/1e9,
                'matrix_flops_2_per_mac_g': 2*matrix_macs/1e9,
                'operator_counts': operators, 'unsupported_operators': unsupported,
                'matrix_coverage_verified': True,
                'convention': 'fvcore: one fused multiply-add = one operation; matrix FLOPs column doubles only counted MACs',
                'coverage': 'All traced convolution/linear/attention matrix products counted; remaining unsupported elementwise/reduction operations disclosed. Not an exact all-operation FLOP count.'}
    except Exception as exc:
        return {'status': 'failed', 'error': str(exc),
                'fvcore_counted_gops_1_fma': None, 'counted_matrix_gmacs': None, 'matrix_flops_2_per_mac_g': None}
    finally:
        torch.backends.mha.set_fastpath_enabled(was_fast)
        clear_bias_cache(model)


def hardware_identity(device):
    info = {'runtime': runtime_identity(), 'device': str(device), 'platform': platform.platform(),
            'cpu': platform.processor(), 'torch_cpu_threads': torch.get_num_threads(),
            'precision': 'FP32', 'batch_size': 1, 'compiled': False,
            'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
            'matmul_allow_tf32': bool(torch.backends.cuda.matmul.allow_tf32),
            'cudnn_allow_tf32': bool(torch.backends.cudnn.allow_tf32)}
    if device.type == 'cuda':
        p = torch.cuda.get_device_properties(device)
        info.update(gpu=p.name, gpu_memory_bytes=p.total_memory, capability=[p.major,p.minor],
                    gpu_uuid=str(getattr(p,'uuid','unavailable')))
    return info


@torch.no_grad()
def benchmark_all(ctx):
    # Deliberately measures ALL arms in ONE invocation on the SAME device.
    # Resuming this phase reruns the whole small timing block; no mixed-GPU table.
    budget(ctx)
    root = ctx['root']/'efficiency'; root.mkdir(exist_ok=True)
    receipt = root/'complete.json'
    if receipt.exists():
        receipt.unlink()  # A partial new benchmark must never retain a completed marker.
    device = ctx['device']; hw = hardware_identity(device)
    bench_spec = {'images': CFG.benchmark_images, 'repeats': CFG.benchmark_repeats,
                  'warmup': CFG.benchmark_warmup, 'hardware': hw}
    save_json(bench_spec, root/'hardware_and_protocol.json')
    sample = ctx['val'].sample(n=min(CFG.benchmark_images,len(ctx['val'])), random_state=548).reset_index(drop=True)
    images = []
    for path in sample.path:
        with Image.open(path) as im:
            images.append(im.convert('RGB').copy())
    tensors = [letterbox(im,CFG.image_size)[:2] for im in images]
    # Preloaded device tensors exclude disk, preprocessing and host->device transfer.
    tensors = [(x[None].to(device),v[None].to(device)) for x,v in tensors]
    def sync():
        if device.type == 'cuda': torch.cuda.synchronize(device)
    order = [(arm,seed) for arm in ARMS for seed in CFG.seeds]
    random.Random(712).shuffle(order)
    rows, samples = [], []
    for arm, seed in order:
        model = load_student(ctx,arm,seed).float()
        bundle = export_student(ctx,model,arm,seed)
        for i in range(CFG.benchmark_warmup):
            model(*tensors[i % len(tensors)])
        sync()
        if device.type == 'cuda':
            baseline_mem = torch.cuda.memory_allocated(device)
            torch.cuda.reset_peak_memory_stats(device)
        forward_ms, decoded_ms = [], []
        for repeat in range(CFG.benchmark_repeats):
            for i,(x,v) in enumerate(tensors):
                sync(); start=time.perf_counter(); model(x,v); sync()
                elapsed=1000*(time.perf_counter()-start); forward_ms.append(elapsed)
                samples.append({'arm':arm,'seed':seed,'scope':'forward','repeat':repeat,'sample_id':sample.iloc[i].sample_id,'milliseconds':elapsed})
            for i,im in enumerate(images):
                sync(); start=time.perf_counter()
                x,v,_=letterbox(im,CFG.image_size)
                model(x[None].to(device),v[None].to(device));sync()
                elapsed=1000*(time.perf_counter()-start);decoded_ms.append(elapsed)
                samples.append({'arm':arm,'seed':seed,'scope':'decoded_image_to_logits','repeat':repeat,'sample_id':sample.iloc[i].sample_id,'milliseconds':elapsed})
        peak = (torch.cuda.max_memory_allocated(device)-baseline_mem)/1e6 if device.type=='cuda' else None
        compute = compute_accounting(model,*tensors[0])
        save_json(compute,root/f'{arm}_seed_{seed}_compute.json')
        row={'arm':arm,'seed':seed,'parameters':sum(p.numel() for p in model.parameters()),
             'checkpoint_mb':bundle.stat().st_size/1e6,'forward_mean_ms':float(np.mean(forward_ms)),
             'forward_median_ms':float(np.median(forward_ms)),'forward_p95_ms':float(np.percentile(forward_ms,95)),
             'decoded_to_logits_mean_ms':float(np.mean(decoded_ms)),
             'decoded_to_logits_p95_ms':float(np.percentile(decoded_ms,95)),
             'additional_peak_cuda_memory_mb':peak,'compute_status':compute['status'],
             'fvcore_counted_gops_1_fma':compute.get('fvcore_counted_gops_1_fma'),
             'counted_matrix_gmacs':compute.get('counted_matrix_gmacs'),
             'matrix_flops_2_per_mac_g':compute.get('matrix_flops_2_per_mac_g'),
             'teacher_at_inference':False,'extra_crops':0,'ensemble':False,'tta':False,
             'hardware_id':object_hash(hw)}
        rows.append(row);print('Efficiency:',arm,seed,f'{row["parameters"]/1e6:.3f}M params, {row["forward_median_ms"]:.2f} ms median',flush=True)
        del model;gc.collect()
        if device.type=='cuda':torch.cuda.empty_cache()
    pd.DataFrame(rows).sort_values(['arm','seed']).to_csv(root/'efficiency_by_run.csv',index=False)
    pd.DataFrame(samples).to_csv(root/'latency_samples.csv',index=False)
    pd.DataFrame(rows).groupby('arm',sort=True).agg(
        parameters=('parameters','first'),checkpoint_mb_mean=('checkpoint_mb','mean'),
        counted_matrix_gmacs=('counted_matrix_gmacs','first'),
        fvcore_counted_gops_1_fma=('fvcore_counted_gops_1_fma','first'),
        forward_ms_mean=('forward_mean_ms','mean'),forward_ms_seed_sd=('forward_mean_ms','std'),
        decoded_to_logits_ms_mean=('decoded_to_logits_mean_ms','mean')).reset_index().to_csv(root/'efficiency_summary.csv',index=False)
    save_json({'status':'complete','contract':ctx['contract_hash'],'runs':len(rows),
               'hardware':hw,'compute_all_succeeded':all(r['compute_status']=='measured' for r in rows),
               'timing_scope':'Batch1 FP32. Forward includes normalization and active attention/cues. '
                              'Decoded-to-logits adds letterboxing and device transfer. Disk decoding excluded. '
                              'Mean +/- seed SD here is timing variation, not uncertainty of classification accuracy.'},receipt)
    return rows

## Plots, reports archive and the complete study

Runs teacher → 12 matched students → validation lock → test/paired confidence intervals → same-device efficiency → paper tables and figures. Session pauses preserve completed jobs. Unexpected errors export available reports and are re-raised.

The reports ZIP excludes `.pt` files. Keep the entire output folder for resume and deployment.

In [15]:
def assemble_paper_table(ctx):
    test_path=ctx['root']/'final_test/test_results_summary.csv'
    if not test_path.exists():return
    test=pd.read_csv(test_path).set_index('arm')
    validation=pd.read_csv(ctx['root']/'validation_results_summary.csv').set_index('arm')
    efficiency=pd.read_csv(ctx['root']/'efficiency/efficiency_summary.csv').set_index('arm')
    cis=pd.read_csv(ctx['root']/'final_test/test_group_bootstrap_ci.csv')
    cis=cis[cis.seed.astype(str)=='mean_of_3'].set_index(['arm','metric'])
    rows=[]
    for arm in ARMS:
        row={'arm':arm,'description':ARMS[arm]['label'],'attention':ARMS[arm]['attention'],
             'SIPaKMeD_KD':ARMS[arm]['kd'],'seeds':int(test.loc[arm,'seeds']),
             'unique_test_images':int(test.loc[arm,'n_unique_images'])}
        for metric in ('accuracy','macro_f1'):
            row['test_'+metric+'_mean_percent']=100*test.loc[arm,metric+'_mean']
            row['test_'+metric+'_seed_sd_pp']=100*test.loc[arm,metric+'_seed_sd']
            row['test_'+metric+'_group_ci_low_percent']=100*cis.loc[(arm,metric),'ci95_low']
            row['test_'+metric+'_group_ci_high_percent']=100*cis.loc[(arm,metric),'ci95_high']
            row['validation_'+metric+'_mean_percent']=100*validation.loc[arm,metric+'_mean']
        row.update(parameters=int(efficiency.loc[arm,'parameters']),
                   counted_matrix_gmacs=efficiency.loc[arm,'counted_matrix_gmacs'],
                   forward_ms_mean=efficiency.loc[arm,'forward_ms_mean'],
                   decoded_to_logits_ms_mean=efficiency.loc[arm,'decoded_to_logits_ms_mean'])
        rows.append(row)
    pd.DataFrame(rows).to_csv(ctx['root']/'paper_ablation_table.csv',index=False)


def plot_results(ctx):
    out=ctx['root']/'figures';out.mkdir(exist_ok=True)
    for arm in ARMS:
        fig,axes=plt.subplots(1,2,figsize=(10,3.7))
        for seed in CFG.seeds:
            path=ctx['root']/arm/f'seed_{seed}'/'history.csv'
            if not path.exists():continue
            h=pd.read_csv(path)
            axes[0].plot(h.epoch,h.train_loss,label=str(seed))
            axes[1].plot(h.epoch,h.macro_f1*100,label=str(seed))
        axes[0].set(xlabel='Epoch',ylabel='Training objective')
        axes[1].set(xlabel='Epoch',ylabel='Validation macro-F1 (%)')
        for ax in axes:ax.legend(title='Seed');ax.grid(alpha=.2)
        fig.suptitle(ARMS[arm]['label']);fig.tight_layout();fig.savefig(out/f'{arm}_learning.png',dpi=150);plt.close(fig)
    test=ctx['root']/'final_test/test_results_summary.csv'
    if test.exists():
        t=pd.read_csv(test).set_index('arm').loc[list(ARMS)].reset_index()
        fig,ax=plt.subplots(figsize=(9,4.6));x=np.arange(len(t));w=.35
        ax.bar(x-w/2,100*t.accuracy_mean,w,yerr=100*t.accuracy_seed_sd,capsize=4,label='Accuracy')
        ax.bar(x+w/2,100*t.macro_f1_mean,w,yerr=100*t.macro_f1_seed_sd,capsize=4,label='Macro-F1')
        ax.set(xticks=x,xticklabels=list(ARMS),ylabel='Percent',ylim=(0,100),
               title='Fixed source-group test: mean +/- seed SD; three models, no ensemble')
        ax.legend();ax.grid(axis='y',alpha=.2);fig.tight_layout();fig.savefig(out/'test_ablation.png',dpi=170);plt.close(fig)


def export_reports(ctx):
    path=ctx['root']/'lightcyto_final_ablation_reports.zip'
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ctx['root'].rglob('*')):
            if p.is_file() and p.suffix.lower() in {'.json','.csv','.png','.svg','.md'}:
                z.write(p,p.relative_to(ctx['root']))
    return path


def run_ablation(ctx):
    teacher=None;model=None
    try:
        source=find_fresh_source(ctx);reuse_original(ctx,source)
        teacher_done=completed_stage(ctx,'teacher',CFG.teacher_epochs)
        seed_all(CFG.teacher_seed)
        teacher=new_teacher(CFG.pretrained and not teacher_done and not (ctx['root']/'teacher/last.pt').exists()).to(ctx['device'])
        fit_job(ctx,'teacher',teacher,CFG.teacher_epochs,CFG.teacher_lr,CFG.teacher_seed,
                train_epoch_function(ctx,is_teacher=True),lambda m:image_validation(m,ctx['val'],ctx['device'],True))
        freeze(teacher)
        teacher_hash=file_hash(ctx['root']/'teacher/best.pt')
        teacher_lock=ctx['root']/'teacher_lock.json'
        expected={'teacher_sha256':teacher_hash,'contract':ctx['contract_hash']}
        if teacher_lock.exists() and read_json(teacher_lock)!=expected:
            raise ValueError('Frozen teacher changed between runs.')
        save_json(expected,teacher_lock)
        validation_rows=[]
        # Fixed job order; every epoch is independently seeded, so order cannot
        # change the declared initialization or augmentation schedule.
        for seed in CFG.seeds:
            for arm,spec in ARMS.items():
                name=f'{arm}/seed_{seed}'
                if completed_stage(ctx,name,CFG.student_epochs):
                    model=load_student(ctx,arm,seed)
                else:
                    model=initialized_student(ctx,seed,arm).to(ctx['device'])
                    fit_job(ctx,name,model,CFG.student_epochs,CFG.student_lr,seed,
                            train_epoch_function(ctx,teacher=teacher,use_kd=spec['kd']),
                            lambda m:image_validation(m,ctx['val'],ctx['device']))
                    freeze(model)
                validation_rows.append(evaluate_run(ctx,model,arm,seed,'val'))
                results_summary(validation_rows,ctx['root']/'validation_results.csv')
                del model;model=None;gc.collect()
                if ctx['device'].type=='cuda':torch.cuda.empty_cache()
        del teacher;teacher=None;gc.collect()
        if ctx['device'].type=='cuda':torch.cuda.empty_cache()
        lock=lock_evaluation(ctx,validation_rows)
        if CFG.run_final_test:
            rows=[];lock_hash=object_hash(lock)
            for arm in ARMS:
                for seed in CFG.seeds:
                    budget(ctx)
                    relative=f'{arm}/seed_{seed}/best.pt'
                    if file_hash(ctx['root']/relative)!=lock['checkpoint_sha256'][relative]:
                        raise ValueError('Locked weights changed: '+relative)
                    model=load_student(ctx,arm,seed)
                    rows.append(evaluate_run(ctx,model,arm,seed,'test',lock_hash))
                    results_summary(rows,ctx['root']/'final_test/test_results.csv')
                    del model;model=None;gc.collect()
                    if ctx['device'].type=='cuda':torch.cuda.empty_cache()
            bootstrap_reports(ctx)
            save_json({'status':'complete','evaluation_lock_hash':lock_hash,'runs':len(rows),
                       'unique_images':len(ctx['test']),'primary_arm':PRIMARY_ARM,
                       'scope':'Post-selection follow-up; fixed source-image groups, not patient-level or external validation'},
                      ctx['root']/'final_test/complete.json')
        efficiency=benchmark_all(ctx)
        assemble_paper_table(ctx)
        plot_results(ctx)
        status={'status':'complete' if CFG.run_final_test else 'complete_test_disabled',
                'completed_student_jobs':count_complete(ctx),'planned_student_jobs':12,
                'test_evaluated':CFG.run_final_test,'primary_arm':PRIMARY_ARM,
                'compute_all_succeeded':all(r['compute_status']=='measured' for r in efficiency)}
    except SessionPause as exc:
        status={'status':'paused','reason':str(exc),'completed_student_jobs':count_complete(ctx),'planned_student_jobs':12,
                'resume':'Save the FULL output, attach it, set resume_source, and Run All.'}
        print(status['resume'],flush=True)
    except Exception as exc:
        save_json({'status':'error','type':type(exc).__name__,'message':str(exc),
                   'completed_student_jobs':count_complete(ctx),'planned_student_jobs':12},ctx['root']/'session_status.json')
        export_reports(ctx)
        raise
    finally:
        del teacher,model;gc.collect()
        if ctx['device'].type=='cuda':torch.cuda.empty_cache()
    save_json(status,ctx['root']/'session_status.json')
    export_reports(ctx)
    return status

## Verify your saved teacher path

Configured input: **My_final_model → lightcyto_tinyvit_end_to_end**.

The screenshot gives an input title and folder name, not a complete filesystem path. This cell finds the actual Kaggle mount, checks the original experiment identity and teacher checkpoint hash, and prints the resolved absolute path. It stops before training if the required weights are missing instead of silently retraining the teacher.

Only the saved teacher is reused; the twelve student ablations remain matched new runs. When resuming this ablation's full output, its already completed teacher takes priority.

In [16]:
# Resolve the attached input shown in your screenshot before starting training.
# This is path setup only; the model, split and scientific contract stay unchanged.
_fresh_hint = CFG.fresh_source
_saved_teacher = any(
    (Path(root)/'ablation_contract.json').is_file()
    and (Path(root)/'teacher/complete.json').is_file()
    and (Path(root)/'teacher/best.pt').is_file()
    for root in (CFG.output_root, CFG.resume_source) if root
)
if _saved_teacher:
    CFG.reuse_verified_fresh = False
    print('An ablation teacher is already saved; its checkpoint will be verified on resume.')
elif CFG.reuse_verified_fresh and _fresh_hint:
    _fresh_direct = Path(_fresh_hint).expanduser()
    _fresh_candidates = []
    if _fresh_direct.is_dir():
        _fresh_candidates = ([_fresh_direct] if (_fresh_direct/'experiment.json').is_file()
                             else [p.parent for p in _fresh_direct.rglob('experiment.json')])
    else:
        # Handles both /kaggle/input/<slug>/... and nested datasets/notebooks mounts.
        for _scan_root in (Path('/kaggle/input'), Path('/kaggle/working')):
            if _scan_root.exists():
                _fresh_candidates.extend(
                    p.parent for p in _scan_root.rglob('experiment.json')
                    if p.parent.name == 'lightcyto_tinyvit_end_to_end'
                )
    _fresh_verified = []
    _fresh_rejected = []
    for _candidate in sorted(set(_fresh_candidates)):
        try:
            if read_json(_candidate/'experiment.json') != ORIGINAL_EXPERIMENT:
                _fresh_rejected.append(str(_candidate)+' (different experiment)')
                continue
            _teacher_file = _candidate/'teacher/best.pt'
            if not _teacher_file.is_file():
                _fresh_rejected.append(str(_candidate)+' (teacher/best.pt missing; reports alone are insufficient)')
                continue
            if file_hash(_teacher_file) != ORIGINAL_TEACHER_SHA:
                _fresh_rejected.append(str(_candidate)+' (teacher checkpoint does not match your reported run)')
                continue
            _fresh_verified.append(_candidate)
        except (OSError, ValueError, TypeError) as _source_error:
            _fresh_rejected.append(str(_candidate)+' ('+str(_source_error)+')')
    if not _fresh_verified:
        raise FileNotFoundError(
            'Cannot verify the saved teacher from My_final_model/lightcyto_tinyvit_end_to_end. '
            'Attach that FULL output containing experiment.json and teacher/best.pt, or set '
            'reuse_verified_fresh=False to intentionally train a new teacher. Checked: '
            + ('; '.join(_fresh_rejected) if _fresh_rejected else 'no matching output folders found')
        )
    # My_final_model is the visible input title; Kaggle may normalize it to my-final-model.
    # Prefer that input. Any fallback must still have the exact original experiment and weight hash.
    _fresh_named = [p for p in _fresh_verified
                    if any(re.sub(r'[^a-z0-9]', '', part.casefold()) == 'myfinalmodel' for part in p.parts)]
    _fresh_pool = _fresh_named or _fresh_verified
    CFG.fresh_source = str(sorted(_fresh_pool, key=lambda p: (len(str(p)), str(p)))[0])
    print('Verified teacher source:', CFG.fresh_source)
    print('Teacher checkpoint:', str(Path(CFG.fresh_source)/'teacher/best.pt'))
    print('Teacher will be reused. All 12 students use the new matched initialization.')
else:
    print('Explicit teacher-source check skipped; the configured training/reuse policy applies.')


Verified teacher source: /kaggle/input/datasets/anindobaruabijoy/my-final-model/lightcyto_tinyvit_end_to_end
Teacher checkpoint: /kaggle/input/datasets/anindobaruabijoy/my-final-model/lightcyto_tinyvit_end_to_end/teacher/best.pt
Teacher will be reused. All 12 students use the new matched initialization.


## Run the complete study

This is the execution cell. `Run All` executes it once. A `paused` status means resume the saved full output; `complete` means all requested stages finished. A compute failure is recorded separately and should be resolved before reporting compute comparisons.

In [17]:
from IPython.display import display, FileLink
CONTEXT = prepare_ablation()
display(pd.read_csv(CONTEXT['root']/'split_summary.csv'))
RESULT = run_ablation(CONTEXT)
print(json.dumps(RESULT, indent=2))
REPORTS_ZIP = export_reports(CONTEXT)
display(FileLink(str(REPORTS_ZIP)))

Exact split: {'train': 2568, 'test': 762, 'val': 719} 
Device: cuda:0 
Student jobs: 12


,split,class_name,images,groups
0,test,Dyskeratotic,159,43
1,test,Koilocytotic,172,51
2,test,Metaplastic,163,55
3,test,Parabasal,124,16
4,test,Superficial-Intermediate,144,27
5,train,Dyskeratotic,514,143
6,train,Koilocytotic,490,150
7,train,Metaplastic,491,174
8,train,Parabasal,536,77
9,train,Superficial-Intermediate,537,76


Verified fresh source: /kaggle/input/datasets/anindobaruabijoy/my-final-model/lightcyto_tinyvit_end_to_end
Reused verified completed stage: teacher


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.6MB            

model.safetensors: downloading bytes:           |  0.00B            

A_PLAIN/seed_42 1/45: val acc 89.71%, F1 90.01%; 39.9s, stage ETA ~29.2 min
A_PLAIN/seed_42 2/45: val acc 93.32%, F1 93.43%; 26.4s, stage ETA ~18.9 min
A_PLAIN/seed_42 3/45: val acc 92.35%, F1 92.46%; 24.3s, stage ETA ~17.0 min
A_PLAIN/seed_42 4/45: val acc 93.74%, F1 93.87%; 26.1s, stage ETA ~17.8 min
A_PLAIN/seed_42 5/45: val acc 95.41%, F1 95.55%; 22.1s, stage ETA ~14.8 min
A_PLAIN/seed_42 6/45: val acc 95.69%, F1 95.83%; 22.0s, stage ETA ~14.3 min
A_PLAIN/seed_42 7/45: val acc 94.99%, F1 95.04%; 21.3s, stage ETA ~13.5 min
A_PLAIN/seed_42 8/45: val acc 93.46%, F1 93.64%; 21.2s, stage ETA ~13.1 min
A_PLAIN/seed_42 9/45: val acc 96.80%, F1 96.88%; 21.3s, stage ETA ~12.8 min
A_PLAIN/seed_42 10/45: val acc 95.55%, F1 95.63%; 21.3s, stage ETA ~12.4 min
A_PLAIN/seed_42 11/45: val acc 94.16%, F1 94.23%; 21.6s, stage ETA ~12.2 min
A_PLAIN/seed_42 12/45: val acc 96.66%, F1 96.79%; 22.1s, stage ETA ~12.1 min
A_PLAIN/seed_42 13/45: val acc 95.27%, F1 95.37%; 21.6s, stage ETA ~11.5 min
A_PLAIN/

/kaggle/working/lightcyto_final_ablation/lightcyto_final_ablation_reports.zip

## Read the final tables

`paper_ablation_table.csv` combines the accuracy/F1 results, separate seed SD/group confidence bounds, parameters and latency. Raw metrics are in the underlying CSV files; the display below converts selected rates to percentages for readability.

Use `final_test/paired_effects.csv` to examine whether attention and KD help under the same protocol. Report negative or uncertain effects as such. This factorial experiment establishes contribution within the specified recipe; it does not itself prove novelty, superiority to external published methods or external generalization.

In [18]:
for relative in ('validation_results_summary.csv', 'final_test/test_results_summary.csv',
                 'final_test/paired_effects.csv', 'efficiency/efficiency_summary.csv',
                 'paper_ablation_table.csv'):
    path = Path(CFG.output_root)/relative
    if path.exists():
        print(relative)
        table = pd.read_csv(path)
        for column in ('accuracy_mean','accuracy_seed_sd','macro_f1_mean','macro_f1_seed_sd'):
            if column in table:
                table[column] = 100*table[column]
        display(table)
    else:
        print('Not created yet:', relative)
status_path = Path(CFG.output_root)/'session_status.json'
if status_path.exists():
    print(status_path.read_text())

validation_results_summary.csv


,arm,seeds,n_unique_images,accuracy_mean,accuracy_seed_sd,macro_f1_mean,macro_f1_seed_sd,balanced_accuracy_mean,aggregation
0,A_PLAIN,3,719,98.284655,0.401495,98.332704,0.391560,0.983657,Three single-model runs on the same images; no...
1,B_KD,3,719,98.099212,0.212451,98.148829,0.202277,0.981786,Three single-model runs on the same images; no...
2,C_ATTENTION,3,719,98.238294,0.350016,98.309076,0.331546,0.983243,Three single-model runs on the same images; no...
3,D_FULL,3,719,98.470097,0.367977,98.507373,0.360236,0.985385,Three single-model runs on the same images; no...


final_test/test_results_summary.csv


,arm,seeds,n_unique_images,accuracy_mean,accuracy_seed_sd,macro_f1_mean,macro_f1_seed_sd,balanced_accuracy_mean,aggregation
0,A_PLAIN,3,762,95.669291,0.347211,95.860013,0.306119,0.960323,Three single-model runs on the same images; no...
1,B_KD,3,762,95.931759,0.347211,96.148655,0.336988,0.962756,Three single-model runs on the same images; no...
2,C_ATTENTION,3,762,96.194226,0.681910,96.423658,0.651342,0.965287,Three single-model runs on the same images; no...
3,D_FULL,3,762,95.975503,0.330264,96.187062,0.301480,0.963626,Three single-model runs on the same images; no...


final_test/paired_effects.csv


...

efficiency/efficiency_summary.csv


,arm,parameters,checkpoint_mb_mean,counted_matrix_gmacs,fvcore_counted_gops_1_fma,forward_ms_mean,forward_ms_seed_sd,decoded_to_logits_ms_mean
0,A_PLAIN,5134297,20.673368,1.257866,1.273205,9.093307,0.108878,10.728069
1,B_KD,5134297,20.673368,1.257866,1.273205,9.097338,0.044008,10.714263
2,C_ATTENTION,5404831,21.761221,1.262345,1.277887,10.822195,0.068294,12.476084
3,D_FULL,5404831,21.761221,1.262345,1.277887,11.093278,0.119127,12.762707


paper_ablation_table.csv


,arm,description,attention,SIPaKMeD_KD,seeds,unique_test_images,test_accuracy_mean_percent,test_accuracy_seed_sd_pp,test_accuracy_group_ci_low_percent,test_accuracy_group_ci_high_percent,validation_accuracy_mean_percent,test_macro_f1_mean_percent,test_macro_f1_seed_sd_pp,test_macro_f1_group_ci_low_percent,test_macro_f1_group_ci_high_percent,validation_macro_f1_mean_percent,parameters,counted_matrix_gmacs,forward_ms_mean,decoded_to_logits_ms_mean
0,A_PLAIN,TinyViT + shared head,False,False,3,762,95.669291,0.347211,92.915854,97.761560,98.284655,95.860013,0.306119,93.286238,97.709667,98.332704,5134297,1.257866,9.093307,10.728069
1,B_KD,TinyViT + shared head + KD,False,True,3,762,95.931759,0.347211,93.351664,97.815898,98.099212,96.148655,0.336988,93.881543,97.828462,98.148829,5134297,1.257866,9.097338,10.714263
2,C_ATTENTION,TinyViT + custom attention,True,False,3,762,96.194226,0.681910,93.630620,98.068314,98.238294,96.423658,0.651342,94.243145,98.061045,98.309076,5404831,1.262345,10.822195,12.476084
3,D_FULL,TinyViT + custom attention + KD,True,True,3,762,95.975503,0.330264,93.121429,98.070961,98.470097,96.187062,0.301480,93.669379,98.038021,98.507373,5404831,1.262345,11.093278,12.762707


{
  "status": "complete",
  "completed_student_jobs": 12,
  "planned_student_jobs": 12,
  "test_evaluated": true,
  "primary_arm": "D_FULL",
  "compute_all_succeeded": true
}


## Resume without losing progress

1. Save/download the entire `/kaggle/working/lightcyto_final_ablation/` output, including `teacher/`, `initialization/`, the four arm folders and all `.pt` files. **The small reports ZIP is insufficient.**
2. Attach that full folder to the next Kaggle notebook session.
3. Set `CFG.resume_source` to the attached folder containing `ablation_contract.json`; keep the writable `output_root` under `/kaggle/working/`.
4. Keep the same training settings and compatible runtime; Run All. Completed jobs are verified and reused; unfinished jobs resume at the last completed epoch. Pretrained initializations and best/last checkpoints remain available.

`fresh_source` and `resume_source` serve different purposes. The former is the earlier fresh model's output (optional teacher reuse); the latter is this notebook's full output (training continuation).

Existing working output takes priority over a read-only resume source and is checked against the contract. Dataset/runtime/code mismatches stop with an explanation. Do not delete a contract or rename incompatible checkpoints to bypass these checks.

## Software validation and reporting limits

Before delivery, the notebook code was exercised locally with real TinyViT/ConvNeXt modules on synthetic images: four arms × three seeds, teacher training, validation lock, test predictions, paired group-bootstrap reports, compute tracing, deployment exports and latency reports. The full attention arm was checked against the original implementation, common initialization tensors were matched, and epoch-boundary resume was checked against uninterrupted dropout/shuffled-batch AdamW training. Known synthetic factorial effects verified the paired statistics.

These are software checks, **not SIPaKMeD results**. Real SIPaKMeD training, official pretrained-weight downloads in Kaggle and CUDA timing were not performed here. Actual results must come from your run. No 97% accuracy target or journal acceptance is guaranteed.

This notebook keeps the previous test partition fixed and discloses its earlier use. It does not add external data, patient identifiers, near-duplicate detection, standard-attention alternatives or a morphology-prior-only ablation. Those limitations remain relevant when writing the paper.

Primary implementation references:

- [TinyViT model card](https://huggingface.co/timm/tiny_vit_5m_224.dist_in22k_ft_in1k)
- [Knowledge distillation](https://arxiv.org/abs/1503.02531)
- [fvcore compute-counting conventions and limitations](https://github.com/facebookresearch/fvcore/blob/main/docs/flop_count.md)
- [Training/validation/test separation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [CLAIM medical-imaging reporting guidance](https://pmc.ncbi.nlm.nih.gov/articles/PMC11304031/)